# Semantic Search phòng trà Hà Nội
## Tài liệu kỹ thuật và thực hành trên Google Colab

**Repository:** [DavidNhac/phong_tra_sematic_search_vip](https://github.com/DavidNhac/phong_tra_sematic_search_vip)

Notebook này được viết lại từ tài liệu cũ và mở rộng theo sơ đồ quy trình người dùng cung cấp. Mục tiêu không chỉ là chạy demo, mà còn giải thích **vì sao từng bước tồn tại, dữ liệu đầu vào/đầu ra là gì, công nghệ được dùng như thế nào, và các quyết định thiết kế ảnh hưởng đến chất lượng tìm kiếm ra sao**.

> **Phân biệt quan trọng:** Notebook có phần mô tả kiến trúc dự án và phần lab chạy độc lập trên Colab. Một số cell thực hành dùng dữ liệu giả lập để minh họa; dữ liệu giả lập không phải dữ liệu đã xác minh về quán thật. Không được diễn giải kết quả demo là benchmark của hệ thống production.

### Liên hệ với sơ đồ quy trình

Sơ đồ được dùng làm khung yêu cầu cho tài liệu:
1. **Xác định bài toán**: mục tiêu khai thác dữ liệu, thông tin cần có, cấu trúc dữ liệu, cách truy vấn và cách hiển thị kết quả.
2. **Tìm nguồn và tải dữ liệu**: website/blog và các nguồn khác; lưu nguồn gốc để có thể truy vết.
3. **Chuyển đổi và chuẩn hóa**: làm sạch HTML/văn bản, chuẩn hóa trường, xử lý dữ liệu thiếu và thống nhất định dạng.
4. **Hợp nhất thực thể trùng lặp**: nhận diện nhiều bài viết cùng nói về một phòng trà mà không làm mất nguồn riêng.
5. **Gán nhãn/tag và tạo metadata**: chỉ gán thuộc tính có căn cứ; phân biệt trường lọc chính xác với mô tả ngữ nghĩa.
6. **Chunk/đơn vị tìm kiếm, embedding và lập chỉ mục**: xác định một point đại diện cho phần mô tả nào; giữ ID quán và nguồn trong payload.
7. **Search**: kết hợp dense/semantic search với sparse/keyword search nếu được triển khai trong code; lọc metadata; gộp kết quả theo quán.
8. **Hiển thị kết quả**: tên, thông tin phù hợp, nguồn và link dẫn về trang gốc.

Sơ đồ là khung tư duy, không phải khẳng định rằng mọi khối đều là một dịch vụ riêng trong code. Các chi tiết implementation phải được đối chiếu với repository hiện hành.

## 1.Mục tiêu học tập

Sau notebook này, người đọc có thể:

- Giải thích luồng từ dữ liệu web đến kết quả tìm kiếm.
- Hiểu vì sao cần hợp nhất thực thể và lưu mô tả theo nguồn.
- Hiểu cách tạo văn bản embedding từ tên, khu vực, tag và mô tả.
- Tạo embedding, lập chỉ mục Qdrant và truy vấn theo ngữ nghĩa.
- Kết hợp dense retrieval và sparse retrieval bằng RRF, đồng thời áp dụng điều kiện lọc metadata.
- Đánh giá kết quả bằng một bộ truy vấn nhỏ, đồng thời hiểu giới hạn của đánh giá thủ công.

### Không nằm trong phạm vi chạy trực tiếp trên Colab

Colab không thể tự truy cập `localhost` trên máy Windows của bạn. Vì thế notebook không tự kết nối SQL Server hoặc Qdrant local của máy cá nhân. Việc crawl toàn bộ website và gọi Gemini cũng không tự chạy mặc định để tránh tốn thời gian, quota hoặc gửi request không chủ ý.

## 2.Kiến trúc dự án

```text
Website / blog
     |
     v
src/crawler/                  Crawl HTML và trích xuất từng mục phòng trà
     |
     v
data/raw/*.json               Dữ liệu thô theo nguồn
     |
     v
src/processing/entity_resolver.py
                               Chuẩn hóa tên, so khớp và hợp nhất quán trùng
     |
     v
src/processing/deterministic_cleaner.py
                               Regex/rule để làm sạch trường
     |
     v
src/processing/llm_cleaner.py
                               Bổ sung/làm sạch bằng LLM (cần cấu hình API)
     |
     v
src/processing/llm_tagger.py
                               Gán tag và kiểm tra bằng chứng
     |
     v
data/processed/tagged_entities.json
     |
     v
src/database/load_sql.py       Nạp dữ liệu quan hệ vào SQL Server
     |
     v
src/embeddings/build_index.py  Đọc quán + mô tả từ SQL Server, tạo text, embedding
     |
     v
Qdrant                         Một point cho mỗi dòng mô tả
     |
     v
src/search/searcher.py          Embed query -> vector search -> filter -> group theo quán
```

**Nguồn sự thật:** theo code `build_index.py`, thông tin quán và mô tả được đọc từ SQL Server. Qdrant giữ vector và payload để phục vụ truy vấn; nó không thay thế hoàn toàn cơ sở dữ liệu quan hệ.

## 2.1. Xác định bài toán trước khi chọn công nghệ

### Bài toán người dùng

Người dùng thường mô tả nhu cầu bằng ngôn ngữ tự nhiên, ví dụ: *“Tìm phòng trà ở Hà Nội có nhạc Trịnh, không gian yên tĩnh, quanh Cầu Giấy.”* Hệ thống cần trả về địa điểm có bằng chứng trong dữ liệu, thay vì chỉ tìm các trang có chứa toàn bộ câu truy vấn.

### Yêu cầu dữ liệu

| Nhóm thông tin | Ví dụ trường | Dùng để làm gì? |
|---|---|---|
| Định danh | `venue_id`, `name`, `aliases` | Nhận diện một quán xuyên suốt pipeline; hỗ trợ gộp kết quả |
| Địa điểm | `address`, `district`, `latitude`, `longitude` | Hiển thị và lọc theo khu vực khi có dữ liệu đáng tin cậy |
| Nội dung | `description`, `embedding_text` | Cung cấp ngữ cảnh để mô hình hiểu ý nghĩa |
| Nhãn chuẩn hóa | `music_genres`, `atmosphere_tags` | Tìm theo thuộc tính rõ ràng và làm giàu văn bản embedding |
| Nguồn | `source_name`, `source_url`, `source_record_id` | Truy vết, kiểm tra độ tin cậy và mở liên kết nguồn |
| Chất lượng | `field_provenance`, `confidence`, `last_seen` nếu có | Biết thông tin đến từ đâu và cần kiểm tra lại hay không |

Không phải mọi trường trong bảng đều chắc chắn đã tồn tại trong mọi bản ghi hiện tại. Đây là **mô hình dữ liệu khuyến nghị để giải thích**; hãy chỉ dùng trường thật sau khi kiểm tra schema và JSON đầu ra của repository.

### Tiêu chí thành công

- Tìm được quán phù hợp cả khi người dùng diễn đạt khác văn bản nguồn.
- Truy vấn có tên riêng/địa chỉ/loại nhạc cụ thể vẫn có khả năng khớp từ khóa.
- Một quán không xuất hiện lặp nhiều lần chỉ vì có nhiều bài viết.
- Mỗi kết quả có nguồn để người dùng tự kiểm chứng.
- Có bộ truy vấn kiểm thử được gán nhãn liên quan; không đánh giá chất lượng chỉ bằng cảm giác.

## 2.2. Mô hình dữ liệu: nguồn, thực thể, mô tả và point tìm kiếm

Nên phân biệt bốn khái niệm, vì chúng không có quan hệ một-một:

1. **Source record**: một mục được lấy từ một trang cụ thể. Hai trang khác nhau có thể cùng nói về một quán.
2. **Venue/entity**: thực thể phòng trà đã được hợp nhất. Đây là danh tính logic của quán.
3. **Description/chunk**: đoạn nội dung cụ thể lấy từ một nguồn hoặc được tách từ nội dung dài. Một quán có thể có nhiều mô tả.
4. **Vector point**: đơn vị được đưa vào vector database để tìm kiếm. Point thường có vector và payload; một quán có thể tương ứng nhiều point.

### Quan hệ đề xuất

```text
source records (nhiều nguồn)
        |
        v
entity resolver / merge
        |
        v
venue entity (một quán chuẩn)
        |
        +---- description từ nguồn A ----> point A + vector
        +---- description từ nguồn B ----> point B + vector
        +---- description từ nguồn C ----> point C + vector
```

### Vì sao không gộp tất cả thành một dòng duy nhất?

- **Giữ nguồn gốc:** có thể biết đoạn mô tả nào nói về đêm nhạc, địa chỉ hoặc không gian.
- **Giảm mất mát thông tin:** nếu nối mọi văn bản thành một đoạn dài, chi tiết từ một nguồn có thể bị chìm trong nội dung khác.
- **Tăng khả năng truy vết:** khi kết quả được trả về, hệ thống có thể hiển thị link nguồn tương ứng.
- **Cho phép gộp ở tầng hiển thị:** tìm kiếm trên nhiều point, sau đó nhóm theo `venue_id` để người dùng thấy một quán một lần.

### Cấu trúc JSON minh họa

Đây là ví dụ khái niệm, không thay thế schema thật của dự án:

```json
{
  "venue_id": "venue_001",
  "name": "Tên phòng trà",
  "aliases": ["Tên gọi khác"],
  "district": "Cầu Giấy",
  "tags": {
    "music_genres": ["nhac_trinh", "acoustic"],
    "atmosphere_tags": ["yen_tinh"]
  },
  "descriptions": [
    {
      "description_id": "desc_001",
      "source_name": "Tên website",
      "source_url": "https://example.com/bai-viet",
      "text": "Đoạn mô tả có căn cứ từ nguồn."
    }
  ]
}
```

Trong vector database, `venue_id`, `description_id`, `source_url`, tag và trường lọc phù hợp thường nằm trong **payload**; vector là mảng số sinh ra từ `embedding_text`. Không đưa API key, thông tin đăng nhập hoặc dữ liệu bí mật vào payload.

## 2.3. Pipeline chi tiết: công nghệ, đầu vào/đầu ra và lý do

| Bước | Đầu vào | Công việc | Đầu ra mong đợi | Công nghệ/khối trong dự án |
|---|---|---|---|---|
| 1. Cấu hình nguồn | URL, quy tắc crawl | Khai báo nguồn được phép lấy và cách xử lý | Danh sách nguồn có cấu hình | `configs/crawler_config.yaml` theo notebook cũ |
| 2. Crawl | HTML/trang web | Tải trang, chọn vùng nội dung liên quan, trích xuất dữ liệu | Bản ghi thô có thông tin nguồn | `src/crawler/` |
| 3. Làm sạch | HTML và text thô | Bỏ menu/footer/nội dung lặp, chuẩn hóa khoảng trắng và ký tự | Text ít nhiễu hơn | Các bước clean/processing hiện có |
| 4. Chuẩn hóa | Tên, địa chỉ, mô tả, trường nguồn | Chuẩn hóa Unicode, tên trường, giá trị và định dạng | Bản ghi có schema nhất quán | processing/cleaning |
| 5. Gộp trùng | Nhiều bản ghi nguồn | Tạo candidate, so sánh tên/alias và các tín hiệu khác, quyết định merge | Một entity chuẩn kèm nhiều nguồn | `src/processing/entity_resolver.py` theo notebook cũ |
| 6. Tagging | Entity và mô tả đã chuẩn hóa | Đề xuất nhãn; kiểm tra tag hợp lệ và bằng chứng | Tag chuẩn hóa kèm căn cứ nếu có | `src/processing/llm_tagger.py` theo notebook cũ |
| 7. Tạo embedding text | Tên, khu vực, tag, mô tả | Ghép văn bản có cấu trúc, không nhúng metadata máy móc | `embedding_text` | `src/embeddings/embedder.py` theo notebook cũ |
| 8. Sinh vector | `embedding_text` | Mã hóa văn bản thành vector số | Dense vector | Sentence Transformers / model cấu hình |
| 9. Lập chỉ mục | Vector + payload | Upsert point vào collection | Collection có thể truy vấn | Qdrant theo tài liệu cũ |
| 10. Search | Query và filter | Dense, sparse/keyword, hợp nhất thứ hạng, lọc và nhóm | Danh sách quán xếp hạng | `src/search/searcher.py` và các module liên quan |
| 11. Trình bày | Kết quả đã xếp hạng | Trả tên, mô tả phù hợp, tag và nguồn | Kết quả có link gốc | API/UI nếu được triển khai |

**Lưu ý về công nghệ:** notebook cũ mô tả Qdrant và Sentence Transformers; sơ đồ tham khảo có ghi ChromaDB. Tài liệu này lấy kiến trúc được mô tả trong repository làm chuẩn, không tự ý thay Qdrant bằng ChromaDB chỉ vì sơ đồ minh họa có tên đó. Trước khi dùng các lệnh chạy production, hãy kiểm tra README, cấu hình và `docs/COMMANDS.md` của đúng commit đang sử dụng.

## 2.4. Làm sạch dữ liệu: xử lý gì và vì sao?

Làm sạch không chỉ là xóa HTML tag. Crawl bài blog thường lấy lẫn menu, breadcrumb, quảng cáo, footer, nút chia sẻ, tiêu đề lặp và nội dung không mô tả địa điểm.

### Các lớp làm sạch

1. **Làm sạch DOM/HTML:** ưu tiên vùng nội dung chính (`article`, `main` hoặc selector phù hợp với website); loại `nav`, `footer`, `header`, quảng cáo, menu và phần tử không liên quan. Selector cần cấu hình theo từng site vì cấu trúc HTML khác nhau.
2. **Chuẩn hóa text:** giải mã HTML entity, chuẩn hóa Unicode về một dạng nhất quán, gộp khoảng trắng/newline thừa, loại ký tự điều khiển.
3. **Loại nhiễu:** bỏ câu/đoạn điều hướng lặp lại và đoạn quá ngắn không mang thông tin. Không xóa đoạn chỉ vì nó ngắn nếu đó là thông tin quan trọng như địa chỉ.
4. **Kiểm tra nội dung:** loại bản ghi không phải địa điểm, tiêu đề danh sách chung hoặc nội dung không có thông tin đủ để xác định quán.
5. **Kiểm tra trường:** chuẩn hóa tên trường, kiểu dữ liệu, URL; giữ `null`/thiếu dữ liệu thay vì tự suy đoán.
6. **Lưu dấu vết:** giữ raw source và thông tin URL ở giai đoạn phù hợp để có thể debug và crawl lại.

### Nguyên tắc an toàn

- Không dùng một regex duy nhất để “xóa mọi thứ” trong HTML; regex có thể làm mất nội dung hữu ích.
- Không tự điền giá, giờ diễn, địa chỉ hoặc thể loại nhạc nếu nguồn không nói rõ.
- Giữ lại dữ liệu thô hoặc snapshot phù hợp để so sánh trước/sau.
- Tạo kiểm thử cho các trang mẫu: title, số đoạn, độ dài text, từ khóa điều hướng còn sót và URL nguồn.

### Chỉ số kiểm tra gợi ý

`raw_text_length`, `clean_text_length`, số đoạn bị loại, tỷ lệ bản ghi rỗng, số bản ghi không phải venue và số URL nguồn duy nhất. Các chỉ số này giúp phát hiện parser đột nhiên lấy toàn bộ menu hoặc bỏ mất phần bài viết.

## 2.5. Gộp dữ liệu trùng lặp (entity resolution)

Một phòng trà có thể xuất hiện trong nhiều bài tổng hợp, với tên viết hoa khác nhau, dấu câu khác nhau hoặc tên gọi rút gọn. Gộp trùng cần phân biệt **“hai bản ghi nói về cùng quán”** với **“hai quán có tên gần giống”**.

### Luồng gộp khái niệm

1. **Chuẩn hóa tên để so sánh:** Unicode, khoảng trắng, chữ hoa/thường, dấu câu và các biến thể tên. Nên giữ tên hiển thị nguyên bản riêng; tên chuẩn hóa chỉ là khóa so sánh.
2. **Tạo ứng viên (candidate generation):** so khớp các bản ghi có tên/alias tương tự, và dùng địa chỉ, website, số điện thoại hoặc khu vực làm tín hiệu bổ sung nếu có.
3. **Chấm điểm tương đồng:** fuzzy string similarity có thể dùng để đề xuất cặp cần xem xét. Ngưỡng không nên được coi là chân lý cho mọi dữ liệu.
4. **Quyết định merge:** tự động gộp khi có bằng chứng đủ mạnh; đưa cặp mơ hồ vào danh sách kiểm tra thay vì gộp liều.
5. **Chọn giá trị chuẩn:** ưu tiên trường có độ tin cậy/nguồn tốt hơn; giữ các alias và các mô tả nguồn thay vì ghi đè mất thông tin.
6. **Tạo ID ổn định:** tất cả bản ghi nguồn được gộp tham chiếu cùng `venue_id`.
7. **Kiểm tra sau gộp:** số entity trước/sau, số bản ghi bị loại vì không phải venue, các cluster có nhiều tên khác nhau và những cặp có nguy cơ gộp nhầm.

### Những gì không nên làm

- Không gộp chỉ vì hai tên cùng chứa “phòng trà”.
- Không loại các mô tả từ nguồn khác sau khi gộp; chúng có thể mang thông tin bổ sung.
- Không coi việc bỏ duplicate URL là đủ: hai URL khác nhau vẫn có thể nói về cùng một quán.
- Không coi việc tên giống nhau là bằng chứng duy nhất nếu dữ liệu có thể có chi nhánh hoặc quán trùng tên.

Trong notebook cũ có nhắc `src/processing/entity_resolver.py` và ngưỡng so khớp tên. Hãy đối chiếu giá trị ngưỡng và thứ tự ưu tiên merge với code hiện hành trước khi mô tả chúng như quy tắc production.

## 2.6. Chunking: đơn vị nào được embed?

Trong dự án dạng tìm kiếm địa điểm, **chunking không nhất thiết là chia mọi văn bản thành các đoạn cố định 500 token**. Đơn vị phù hợp phụ thuộc dữ liệu đã trích xuất.

Notebook cũ mô tả `tea_room_descriptions`: mỗi mô tả gắn với một nguồn có thể trở thành một point riêng. Đây gần với **source-level description chunking** hơn là chia tài liệu dài theo độ dài token.

### Quy tắc thiết kế

- Nếu một nguồn đã có một đoạn mô tả ngắn, tập trung vào một quán: giữ đoạn đó làm đơn vị tìm kiếm.
- Nếu một bài rất dài liệt kê nhiều quán: tách theo từng mục/quán trước, tránh tạo chunk trộn thông tin của nhiều địa điểm.
- Nếu mô tả của một quán dài và chứa nhiều chủ đề độc lập: có thể tách theo đoạn/tiêu đề, nhưng mỗi chunk phải giữ `venue_id`, `source_url`, `description_id` và thứ tự/tiêu đề nếu có.
- Nếu nhiều nguồn nói về cùng quán: giữ nhiều description point khi chúng mang thông tin khác nhau; khi trả kết quả, nhóm theo `venue_id`.
- Nếu đoạn quá ngắn hoặc chỉ có menu/nội dung điều hướng: loại hoặc gắn trạng thái không đủ chất lượng trước khi embedding.

### Vì sao giữ metadata khi chunk?

Sau khi chunk, đoạn text không tự cho biết nó thuộc quán nào hay đến từ URL nào. Payload cần giữ ID quán và nguồn để hệ thống có thể nhóm kết quả, hiển thị link và kiểm tra dữ liệu.

### Khi nào dùng overlap?

Overlap hữu ích khi chia văn bản dài liên tục và ý nghĩa bị cắt ngang ranh giới. Với các mục địa điểm đã tách theo từng quán hoặc mô tả nguồn ngắn, overlap thường không cần thiết và có thể tạo các point gần như trùng nhau. Vì vậy không nên áp dụng overlap mặc định nếu cấu trúc nội dung đã cung cấp ranh giới tự nhiên.

**Kiểm tra trước khi lập chỉ mục:** mỗi chunk nên có một chủ đề chính, không trộn nhiều venue, có nguồn truy vết, và `embedding_text` đủ ngữ cảnh để hiểu khi đứng riêng.

## 2.7. Tag và metadata: hai vai trò khác nhau

### Tag là nhãn ngữ nghĩa có tập giá trị được kiểm soát

Ví dụ: `nhac_trinh`, `acoustic`, `jazz`, `yen_tinh`. Dùng mã chuẩn hóa trong dữ liệu để tránh các biến thể như “Nhạc Trịnh”, “nhạc trịnh”, “Trịnh” trở thành các nhãn khác nhau; khi hiển thị hoặc tạo văn bản embedding, đổi mã thành nhãn dễ đọc.

### Metadata là các trường có cấu trúc để truy vết, lọc và hiển thị

Ví dụ: `venue_id`, `district`, `address`, `source_url`, `source_name`, `price_min_vnd` (nếu có căn cứ), `opening_hours` (nếu có căn cứ). Metadata có thể bao gồm tags, nhưng hai khái niệm không đồng nghĩa.

| Loại | Ví dụ | Dùng ở đâu? |
|---|---|---|
| Nội dung tự do | mô tả không gian, chương trình biểu diễn | Dense embedding và hiển thị |
| Tag chuẩn hóa | `music_genres=["nhac_trinh"]` | Lọc chính xác, tạo text embedding |
| Trường lọc | `district="Cầu Giấy"` | Điều kiện lọc có cấu trúc |
| Truy vết | `source_url`, `source_name` | Link nguồn, kiểm tra độ tin cậy |
| Khóa định danh | `venue_id`, `description_id` | Gộp kết quả, tránh lặp quán |

### Chiến lược gán tag trong tài liệu cũ

Notebook trước mô tả chiến lược LLM đề xuất tag kết hợp luật/Regex kiểm chứng trong `src/processing/llm_tagger.py`. Ý tưởng:
1. Chỉ cho phép tag nằm trong taxonomy được định nghĩa.
2. Dùng LLM để đề xuất từ nội dung, không tự phát minh nhãn ngoài danh sách.
3. Kiểm tra bằng chứng trong văn bản; loại tag không có căn cứ theo luật hiện có.
4. Hạn chế tag quá mức và tránh nhãn mâu thuẫn.
5. Lưu tag dưới dạng có cấu trúc, không chỉ chèn một chuỗi không phân tích được vào mô tả.

Đây là mô tả theo tài liệu cũ; mức độ kiểm chứng thực tế cần xác nhận từ code hiện tại. Regex không thể hiểu mọi ngữ cảnh và có thể bỏ sót cách diễn đạt tương đương, vì vậy nên đánh giá precision/recall tag trên mẫu được con người kiểm tra.

### Nên gán tag gì?

Ưu tiên thuộc tính có giá trị trực tiếp cho truy vấn:
- **Thể loại/loại hình âm nhạc:** nhạc Trịnh, acoustic, jazz, bolero, piano… chỉ khi có bằng chứng.
- **Không gian/trải nghiệm:** yên tĩnh, lãng mạn, sân vườn… dùng thận trọng vì có tính chủ quan.
- **Thông tin thực tế:** địa chỉ, quận, giờ biểu diễn, giá… nên giữ thành trường riêng, không chỉ biến thành tag.
- Không suy diễn giờ mở cửa, mức giá hay lịch diễn từ nội dung không xác nhận.

## 2.8. Xây dựng `embedding_text`: tại sao không embed payload nguyên xi?

Mô hình embedding nhận văn bản, không tự động đọc toàn bộ payload như database. Vì vậy, pipeline cần chọn những trường hữu ích và ghép thành văn bản có ngữ cảnh.

Ví dụ minh họa:

```text
Tên địa điểm: Phòng trà ...
Khu vực: Cầu Giấy, Hà Nội
Thể loại nhạc: nhạc Trịnh, acoustic
Không gian: yên tĩnh
Mô tả: Có các đêm nhạc ... (nội dung từ nguồn)
```

### Nguyên tắc

- Đưa tên quán, khu vực, tag dễ hiểu và mô tả có ích vào văn bản embedding.
- Không đưa toàn bộ URL, ID nội bộ, timestamp hoặc metadata kỹ thuật vào văn bản embedding trừ khi có lý do rõ ràng.
- Giữ `source_url` và các trường lọc trong payload để sử dụng đúng mục đích.
- Tránh lặp tag quá nhiều lần; lặp từ có thể làm lệch biểu diễn.
- Chuẩn hóa truy vấn và văn bản nhất quán về Unicode/whitespace. Việc chuẩn hóa chữ hoa/thường phục vụ tìm kiếm cần được áp dụng đúng ở các nhánh liên quan; không nên tùy tiện biến đổi dữ liệu hiển thị gốc.
- Query embedding phải dùng cùng model và cùng không gian vector với document embedding.

Model nhỏ dùng trong lab nhằm giảm thời gian khởi động. Không được so sánh điểm số giữa các model khác nhau như thể chúng cùng một thang đo.

## 2.9. Search: dense, sparse, hybrid, metadata và gộp kết quả

### Dense / semantic search

Câu truy vấn được mã hóa thành vector bằng cùng model; Qdrant tìm các vector gần nhất theo metric cấu hình. Nhánh này hữu ích khi người dùng diễn đạt khác nguồn nhưng ý nghĩa gần nhau.

### Sparse / keyword search

Nhánh từ khóa ưu tiên các từ/cụm từ xuất hiện trong nội dung. Cách triển khai có thể là BM25, sparse vectors hoặc cơ chế keyword khác. Không được khẳng định dự án đang dùng BM25/sparse vectors nếu code hiện tại không thể hiện điều đó.

### Hybrid Search

Hybrid retrieval thực sự cần chạy hoặc lấy kết quả từ ít nhất hai tín hiệu tìm kiếm (chẳng hạn dense và sparse/keyword), sau đó hợp nhất/xếp hạng. Có thể dùng Reciprocal Rank Fusion (RRF) hoặc một cách kết hợp điểm số khác. Không nên cộng trực tiếp các điểm raw nếu chúng có thang đo khác nhau.

RRF thường có dạng:

\[
\operatorname{RRF}(d)=\sum_{r \in R}\frac{1}{k+\operatorname{rank}_r(d)}
\]

Trong đó `rank_r(d)` là thứ hạng tài liệu ở nhánh `r`, còn `k` là hằng số làm trơn. Công thức chỉ minh họa một chiến lược hợp nhất; hãy dùng đúng thuật toán trong code dự án.

### Metadata filtering

Điều kiện như quận hoặc thể loại nhạc có thể dùng để thu hẹp candidate set nếu trường đã có dữ liệu đủ tin cậy. Filter chính xác khác với semantic relevance: một quán gần nghĩa nhưng sai quận có thể bị loại nếu filter là bắt buộc.

### Chuẩn hóa chữ hoa/thường

Tên và truy vấn cần được chuẩn hóa nhất quán để “JAZZ”, “Jazz” và “jazz” không bị xem như ba dạng khác nhau trong nhánh so khớp chữ. Chuẩn hóa so sánh không có nghĩa là thay đổi tên gốc hiển thị. Nếu dự án xử lý cả dấu tiếng Việt, cần kiểm thử riêng trường hợp có dấu/không dấu; không suy ra rằng đã hỗ trợ bỏ dấu chỉ từ việc không phân biệt hoa thường.

### Gộp kết quả theo quán

Một quán có nhiều description point có thể chiếm nhiều vị trí trong top-k. Sau retrieval, nhóm theo `venue_id` (hoặc ID thực thể tương ứng), giữ điểm/đoạn đại diện theo quy tắc rõ ràng, và giữ link nguồn tương ứng. Nên lấy đủ candidate ở mỗi nhánh trước khi gộp, nếu không một quán có nhiều point có thể chiếm chỗ của các quán khác.

### Luồng truy vấn đề xuất

```text
Query người dùng
   ├── normalize query
   ├── dense query embedding → dense retrieval
   └── keyword/sparse query → lexical retrieval
                    |
            fusion / reranking
                    |
       metadata constraints (theo thiết kế)
                    |
      group by venue_id / deduplicate
                    |
     tên + thông tin + điểm phù hợp + source URL
```

Thứ tự filter/fusion có thể khác tùy implementation và yêu cầu. Hãy đối chiếu `src/search/` trước khi mô tả đây là thứ tự chạy chính xác của repository.

## 2.10. Thiết kế kết quả hiển thị và link nguồn

Kết quả nên hữu ích và có thể kiểm chứng. Một kết quả gợi ý nên chứa:
- Tên quán chuẩn và alias cần thiết.
- Mô tả ngắn liên quan nhất đến truy vấn, không phải đoạn text bất kỳ.
- Tag/thuộc tính liên quan, được phân biệt với dữ kiện xác thực.
- Địa chỉ, quận, giờ/giá chỉ khi có dữ liệu đáng tin cậy.
- Link nguồn của đoạn mô tả đã giúp kết quả khớp.
- Điểm/thứ hạng nếu có, kèm ghi chú rằng điểm similarity không phải xác suất đúng.

Nếu nhiều nguồn cùng mô tả một quán, có thể trả một kết quả quán với một nguồn đại diện và danh sách nguồn bổ sung. Không tạo link giả; link phải đến từ payload/source record đã lưu.

## 2.11. Kiểm thử và đánh giá chất lượng

Một số truy vấn mẫu nên được gán nhãn trước khi chỉnh model hoặc trọng số:
- Truy vấn theo nghĩa: “nơi nghe nhạc nhẹ nhàng để thư giãn”.
- Truy vấn có thể loại cụ thể: “phòng trà nhạc Trịnh”.
- Truy vấn theo tên riêng.
- Truy vấn theo địa điểm/quận.
- Truy vấn có nhiều điều kiện.
- Truy vấn có biến thể hoa/thường và dấu tiếng Việt.
- Truy vấn không có kết quả phù hợp, để kiểm tra hệ thống có trả về kết quả không liên quan hay không.

Chỉ số gợi ý:
- **Hit@K:** có ít nhất một kết quả liên quan trong top-K hay không.
- **Precision@K:** tỷ lệ kết quả trong top-K được đánh giá là liên quan.
- **Recall@K:** tỷ lệ các kết quả liên quan đã biết được tìm thấy trong top-K.
- **MRR:** vị trí của kết quả liên quan đầu tiên.
- **Duplicate rate:** mức độ kết quả bị lặp cùng một quán.
- **Source coverage:** tỷ lệ kết quả có link nguồn kiểm chứng được.

Tập demo trong notebook hiện tại chỉ minh họa cách tính; để báo cáo đồ án, cần tạo bộ query–relevance judgments từ dữ liệu thực và ghi rõ cách đánh giá.

## 2.12. Xử lý ảnh trong dự án (đối chiếu với code repository)

Phần này mô tả **đúng những gì repo `DavidNhac/phong_tra_sematic_search_vip` đang làm với ảnh**, theo từng giai đoạn của pipeline. Nội dung được đối chiếu với code ở nhánh `main` (các module `base_crawler.py`, `blog_crawler.py`, `entity_resolver.py`, `sql_models.py`, `load_sql.py`, `build_index.py`, `schema.py`). Dữ liệu `data/raw/` và `data/processed/` không được commit lên GitHub, nên các con số cụ thể (số quán có ảnh, số ảnh) phải lấy từ output của pipeline trên máy phát triển, không suy ra từ repo.

### 2.12.1. Nguyên tắc thiết kế

1. **Chỉ lưu URL ảnh, không tải file ảnh về.** Ảnh được hiển thị bằng cách trỏ thẳng tới URL gốc trên website nguồn (hotlink). Cách này nhẹ, không phải lưu trữ file và không sao chép nội dung, nhưng ảnh có thể đổi link, bị xóa hoặc bị chặn hotlink.
2. **Ảnh gắn theo từng quán, theo vị trí trong DOM.** Với bài dạng danh sách, mỗi quán là một mục dưới một heading; ảnh chỉ được lấy từ các khối nằm giữa heading đó và heading kế tiếp. Nhờ vậy ảnh banner, sidebar hoặc ảnh của quán khác không bị gán nhầm cho mọi quán.
3. **Lọc bằng heuristic, không nhận diện bằng thị giác máy tính.** Repo không phân tích nội dung ảnh (không dùng mô hình thị giác, không kiểm tra ảnh có đúng là quán đó hay không). Việc loại logo, icon, banner dựa trên tên file, đường dẫn và kích thước khai báo trong HTML.
4. **Lọc nhiều lớp.** Một lớp ở crawler (khi trích xuất) và một lớp ở bước gộp trùng (khi chuẩn hóa), vì mỗi lớp bắt được loại rác khác nhau.
5. **Giữ HTML thô để chạy lại offline.** Crawler lưu HTML vào `data/raw_html/`, và `parse_page()` có thể parse lại HTML đã lưu. Khi sửa luật lọc ảnh chỉ cần parse lại, không phải tải lại trang.
6. **Ảnh chỉ phục vụ hiển thị, không tham gia tìm kiếm.** Ảnh không được embed; tìm kiếm ngữ nghĩa chạy hoàn toàn trên văn bản. `image_url` chỉ đi theo kết quả như một trường thông tin.

### 2.12.2. Luồng xử lý ảnh qua từng giai đoạn

```text
HTML bài viết (httpx hoặc Playwright, có retry khi 403/429/5xx)
   │  lưu bản thô: data/raw_html/<nguồn>.html
   ▼
BlogListCrawler.parse_page()            src/crawler/blog_crawler.py
   │  mỗi heading quán -> các khối anh em phía sau (_section_blocks)
   │  extract_section_images(): tối đa 3 ảnh/quán
   ▼
data/raw/<nguồn>.json                   image_url (ảnh đầu) + image_urls
   ▼
EntityResolver.normalize_item()         src/processing/entity_resolver.py
   │  giữ URL bắt đầu bằng http và không khớp _JUNK_IMAGE
   │  gộp trùng: hợp nhất ảnh các nguồn, bỏ trùng URL, giữ tối đa 5
   ▼
resolved_entities.json -> ... -> tagged_entities.json
   ▼
load_sql.upsert_entity()                src/database/load_sql.py
   │  tea_room_images (nhiều dòng) + tea_rooms.image_url (ảnh chính)
   ▼
build_index.py -> Qdrant payload        image_url (ảnh chính) cho mỗi point
   ▼
searcher -> kết quả                     payload.image_url đi kèm kết quả
```

### 2.12.3. Giai đoạn 1: Crawl và trích xuất ảnh theo section

**a) Xác định vùng ảnh của một quán.** `BlogListCrawler._section_blocks()` lấy các phần tử anh em đứng sau heading quán, dừng khi gặp heading cùng cấp hoặc cao hơn, hoặc `footer`. Các thẻ `script`, `style`, `nav`, `aside`, `form`, `noscript`, `button`, `iframe` bị bỏ qua, và các khối có `class`/`id` chứa các từ như `related`, `comment`, `share`, `social`, `sidebar`, `advert`, `breadcrumb`, `newsletter`, `subscribe` cũng bị bỏ. Nếu heading nằm trong một container, code dùng container đó làm điểm bắt đầu.

**b) Tìm thẻ ảnh.** `extract_section_images()` kiểm tra từng khối: nếu bản thân khối là `<img>` thì dùng luôn, rồi thêm mọi `<img>` con (`find_all("img")`). Việc kiểm tra riêng khối là cần thiết vì có website (ví dụ mytour) đặt `<img>` trực tiếp cạnh heading và đoạn văn, mà `find_all()` chỉ nhìn các phần tử con.

**c) Chọn URL tốt nhất từ thẻ.** `_image_src()` ưu tiên theo thứ tự:
1. Thuộc tính lazy-load: `data-src`, `data-lazy-src`, `data-original` (nhiều trang để `src` là ảnh giữ chỗ, ảnh thật nằm ở thuộc tính này);
2. `srcset` / `data-srcset`: lấy phần tử cuối (thường là bản độ phân giải lớn nhất);
3. `src`.

Sau đó `get_absolute_url()` đổi URL tương đối thành tuyệt đối bằng `urljoin` với URL trang, và bỏ các URL bắt đầu bằng `data:` hoặc `javascript:`.

**d) Bộ lọc `_is_good_image()`.** Một ảnh bị loại nếu:
- phần mở rộng (sau khi bỏ query string) là `.svg`, `.gif`, `.ico`;
- tên file chứa một trong các token `logo`, `avatar`, `banner`, `icon`, `sprite`, `placeholder`, `pixel`, `tracking`, `emoji`, `loading`, và token đó được ngăn cách bằng `-`, `_`, `.` hoặc chữ số (ví dụ `site-logo.png` bị loại, nhưng `uploads` thì không);
- đường dẫn có các mẫu của widget bán vé/sản phẩm nhúng trong bài blog: `booking-static`, `/tours/`, `ve-vao-cua`, `voucher`, `combo`;
- thuộc tính `width`, `data-width`, `height`, `data-height` là số nguyên nhỏ hơn 100 (ảnh quá nhỏ thường là icon hoặc pixel theo dõi).

**e) Giới hạn và kết quả.** Ảnh được khử trùng lặp trong cùng một quán và dừng ở tham số `max_images` của crawler (mặc định 3). Mỗi bản ghi thô có `image_url` (ảnh đầu tiên hoặc chuỗi rỗng) và `image_urls` (danh sách). Log của crawler in số mục có ảnh (`with_images`) để theo dõi chất lượng từng nguồn.

### 2.12.4. Giai đoạn 2: Chuẩn hóa và gộp trùng ảnh

`EntityResolver.normalize_item()` lấy danh sách ảnh từ `image_urls` (hoặc `image_url` nếu không có danh sách), chỉ giữ chuỗi bắt đầu bằng `http` và **không** khớp biểu thức `_JUNK_IMAGE` (`booking-static`, `/tours/`, `ve-vao-cua`, `voucher`, `combo`, `/ads?/`, `banner`, `logo`). Lưu ý lớp lọc này so khớp chuỗi con trên toàn bộ URL (không chỉ tên file), nên chặt hơn lớp ở crawler.

Khi nhiều bản ghi từ các nguồn khác nhau được nhận ra là cùng một quán, ảnh được **hợp nhất** theo thứ tự gặp, bỏ URL trùng chính xác, rồi cắt còn tối đa 5 ảnh. Kết quả ghi vào thực thể: `image_url` là ảnh đầu tiên, `image_urls` là tối đa 5 ảnh. Thống kê cuối của resolver in `With images: n/N` để biết bao nhiêu quán có ít nhất một ảnh.

Điều này khớp với nguyên tắc chung của dự án khi gộp dữ liệu: không vứt thông tin của nguồn khác (mô tả, ảnh), nhưng giữ giới hạn để dữ liệu gọn.

### 2.12.5. Giai đoạn 3: Lưu trữ trong SQL Server

Schema có hai tầng:

| Bảng/cột | Vai trò |
|---|---|
| `tea_rooms.image_url` (`NVARCHAR(1000)`) | Ảnh đại diện, tiện truy vấn nhanh và đưa vào payload |
| `tea_room_images` (`id`, `tea_room_id` FK `ON DELETE CASCADE`, `image_url`, `is_primary`, `order_index`) | Toàn bộ ảnh của quán; `order_index = 0` và `is_primary = true` là ảnh chính |

`load_sql.upsert_entity()` làm như sau:
1. Gom `image_url` và các phần tử `image_urls` thành một danh sách, bỏ trùng và giữ nguyên thứ tự;
2. Với mỗi URL, hàm `_url()` **bỏ hẳn** URL dài hơn giới hạn cột (1000 ký tự) thay vì cắt, vì URL bị cắt là URL hỏng;
3. Ghi từng URL vào `tea_room_images` với `order_index` theo vị trí và `is_primary` cho ảnh đầu;
4. Gán `tea_rooms.image_url` bằng ảnh đầu tiên (hoặc `NULL` nếu không có).

Các bảng con (bí danh, mô tả, ảnh, nguồn, tag) được **xóa và tạo lại** mỗi lần nạp một quán, còn quán được nhận diện bằng `slug`, nên nạp nhiều lần không nhân đôi dữ liệu. Hệ quả cần nhớ: chạy lại `load_sql` với file dữ liệu mới sẽ thay thế toàn bộ danh sách ảnh của quán đó bằng ảnh trong file mới.

### 2.12.6. Giai đoạn 4: Qdrant và kết quả tìm kiếm

`build_index.py` đọc quán và mô tả từ SQL Server rồi tạo payload cho từng point; payload có trường `image_url` lấy từ `tea_rooms.image_url` (chỉ ảnh chính). Model payload `PhongTraPayload` khai báo `image_url` là trường tùy chọn.

`searcher.py` trả về kết quả kèm payload, do đó giao diện hoặc API có thể lấy `payload.image_url` để hiển thị ảnh đại diện. Danh sách ảnh đầy đủ (tối đa 5) hiện nằm ở bảng `tea_room_images` trong SQL Server, không nằm trong Qdrant; muốn hiển thị thư viện ảnh cần truy vấn bảng này theo `tea_room_id`. Trình in kết quả dòng lệnh của searcher không in ảnh.

### 2.12.7. Những gì repo không làm (giới hạn cần nêu rõ khi báo cáo)

- **Không kiểm tra URL ảnh còn sống**: chưa có bước gửi request để xác nhận ảnh trả về HTTP 200 và đúng loại ảnh, nên URL chết hoặc bị chặn hotlink vẫn có thể nằm trong dữ liệu.
- **Không tải ảnh về hoặc lưu bản sao**: phụ thuộc hoàn toàn vào website nguồn.
- **Không xác nhận ảnh đúng quán**: ảnh được gán theo vị trí trong DOM, nên bài viết có bố cục lạ (ảnh đặt trước heading, ảnh dùng chung nhiều mục) có thể gán sai.
- **Không dùng thị giác máy tính**: không nhận diện logo, ảnh mờ, ảnh trùng nội dung nhưng khác URL (khử trùng lặp chỉ theo URL).
- **Không dùng Google Places để lấy ảnh**: module `google_places.py` chỉ ghi `place_id` (theo điều khoản của Google) để đối chiếu thông tin, không lấy hoặc lưu ảnh.
- **Ảnh không tham gia embedding hay xếp hạng**: chất lượng tìm kiếm không phụ thuộc ảnh.

### 2.12.8. Công cụ debug và cách đánh giá chất lượng ảnh

- `scripts/debug_image.py` tải một trang, tìm heading chứa tên quán và in HTML của 10 phần tử anh em kế tiếp, kèm `src`, `data-src`, `data-lazy-src` và `alt` của từng `<img>`. Dùng khi một quán không có ảnh để xem ảnh nằm ở thẻ/thuộc tính nào.
- Chỉ số nên theo dõi: tỷ lệ quán có ảnh (`With images` của resolver), số ảnh trung bình mỗi quán, số quán có đủ 3 ảnh trở lên, số URL bị bỏ vì quá dài và tỷ lệ URL ảnh trả về HTTP 200 khi kiểm tra thủ công một mẫu.
- Khi kiểm thử thủ công, nên mở ngẫu nhiên một mẫu quán để xác nhận ảnh đúng địa điểm, vì heuristic không thể tự đảm bảo điều đó.

### 2.12.9. So sánh với phần lab ảnh trên Colab (mục 6)

| Khía cạnh | Repo (pipeline thật) | Lab Colab (mục 6.3 và 6.5.1) |
|---|---|---|
| Vùng lấy ảnh | Các khối anh em sau heading quán, loại khối nhiễu theo class/id | Duyệt `<img>` theo thứ tự DOM, gắn vào section gần nhất |
| Chọn URL | Lazy-load, `srcset` (phần tử cuối), `src` | Tương tự, thêm `<picture><source>` và chọn ảnh có width lớn nhất |
| Lọc | Token tên file, mẫu widget vé, kích thước, đuôi file; thêm `_JUNK_IMAGE` ở resolver | Token tên file, kích thước; thêm bỏ ảnh dùng chung ở nhiều quán |
| Kiểm tra URL | Không | Có (GET kèm `Referer`, kiểm tra `Content-Type` và magic bytes) |
| Số ảnh tối đa | 3 mỗi nguồn, 5 sau gộp | 5 mỗi quán sau kiểm tra |
| Lưu trữ | `tea_room_images` + `tea_rooms.image_url` + payload Qdrant | `tagged_entities.json`, `venue_images.csv` |

Lab Colab là bản rút gọn để học và thử nghiệm, không thay thế các module trong `src/`; khi báo cáo, nên nêu rõ phần nào là của pipeline thật và phần nào chỉ là minh họa.


### 2.1 cấu trúc dữ liệu.

luồng xử lý câu hỏi


Câu hỏi người dùng:

“Phòng trà nhạc Trịnh yên tĩnh ở Cầu Giấy”

-> Dense Search : Hiểu ngữ nghĩa và cách diễn đạt tương đương.


-> Sparse Search: Khớp từ khóa và cụm từ cụ thể.



->RRF Fusion: Hợp nhất thứ hạng từ hai nhánh tìm kiếm

-> Metadata filters + Group by quán: Quận, giá, tag và loại bỏ kết quả trùng quán

->Danh sách phòng trà phù hợp

## 3.Thành phần trong repository và vai trò

| Thành phần | Trách nhiệm |
|---|---|
| `configs/crawler_config.yaml` | Bật/tắt nguồn, URL, parser và tùy chọn trích xuất |
| `src/crawler/` | Tải trang, phân tích HTML, lưu tên, văn bản, ảnh và metadata nguồn |
| `src/processing/entity_resolver.py` | Hợp nhất các mục có khả năng cùng một quán; hạn chế gộp nhầm chi nhánh |
| `src/processing/field_extractors.py` | Chuẩn hóa tên, địa chỉ, quận, giá và giờ |
| `src/processing/deterministic_cleaner.py` | Làm sạch dựa trên quy tắc xác định |
| `src/processing/llm_cleaner.py` | Dùng LLM cho một số trường còn thiếu/khó xử lý |
| `src/processing/llm_tagger.py` | Gán taxonomy tag và xác thực bằng chứng |
| `src/database/sql_models.py`, `sql_client.py`, `load_sql.py` | Schema và thao tác SQL Server |
| `src/embeddings/embedder.py` | Chuẩn hóa văn bản, dựng text và tạo embedding |
| `src/embeddings/build_index.py` | Đọc SQL Server, tạo payload, embedding và upsert vào Qdrant |
| `src/database/indexer.py` | Tạo collection Qdrant và payload indexes |
| `src/search/searcher.py` | Tạo vector cho câu hỏi, lọc metadata, group kết quả theo quán |
| `src/api/` | Lớp API; cần kiểm tra endpoint thực tế trước khi mô tả chi tiết |
| `tests/` | Unit tests cho các phần như embedder, extractors, loader và searcher |

Lệnh pipeline thực tế được ghi trong `docs/COMMANDS.md`. Hãy chạy lệnh từ thư mục gốc repository và kiểm tra log/đầu ra sau từng giai đoạn.

## 4.Chuẩn bị môi trường Colab

Runtime đề xuất: Python 3.11+; GPU giúp tăng tốc embedding. Lab mặc định dùng model nhỏ hơn để lần chạy đầu nhẹ hơn. Model triển khai trong cấu hình dự án là `BAAI/bge-m3` (vector 1024 chiều), có thể tốn nhiều RAM/VRAM và thời gian tải hơn.

**Cách chọn runtime:** Colab → Runtime → Change runtime type → chọn GPU nếu có. Nếu dùng CPU, hãy giữ tập dữ liệu minh họa nhỏ.

In [ ]:
%pip -q install "sentence-transformers>=3.2" "qdrant-client>=1.12,<1.13" pandas numpy

In [ ]:
import os
import json
import time
import unicodedata
from collections import defaultdict

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 5.Chọn model embedding

- `BAAI/bge-m3`: model được khai báo trong cấu hình dự án, embedding dense có 1024 chiều; có khả năng hỗ trợ các biểu diễn sparse/multilingual trong hệ sinh thái model, nhưng **việc dùng model này không tự động làm searcher thành hybrid**.
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`: model nhỏ hơn, phù hợp để chạy demo nhanh trên Colab; vector 384 chiều. Kết quả demo bằng model này không thể xem là kết quả benchmark của model production.

Để chạy nhanh lần đầu, giữ `USE_PRODUCTION_MODEL = False`. Chuyển sang `True` nếu muốn thử model của dự án và runtime đủ tài nguyên.

In [ ]:
USE_PRODUCTION_MODEL = False

if USE_PRODUCTION_MODEL:
    MODEL_NAME = "BAAI/bge-m3"
else:
    MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("Model:", MODEL_NAME)
print("Device:", DEVICE)

t0 = time.time()
embedder = SentenceTransformer(MODEL_NAME, device=DEVICE)
VECTOR_SIZE = embedder.get_sentence_embedding_dimension()
print(f"Vector dimension: {VECTOR_SIZE}; model loaded in {time.time()-t0:.1f}s")

## 6. Crawl → Clean → Extract → Deduplicate → Tag (thực hành trên Colab)

Phần này bổ sung pipeline dữ liệu trước khi embedding. Nó dùng các nguồn đã khai báo trong `configs/crawler_config.yaml` của repository làm điểm xuất phát, nhưng là **bản lab rút gọn viết riêng cho Colab**, không phải chạy nguyên xi các module `src/crawler/` của repo.

Luồng thực hành:

1. Crawl trang bài viết theo danh sách URL và lưu HTML/text thô.
2. Loại các vùng giao diện phổ biến như `nav`, `footer`, `script`, quảng cáo; lấy các heading và đoạn văn liên quan.
3. Trích xuất ứng viên địa điểm từ heading/section; đây là bước heuristic nên cần xem lại kết quả.
4. Gộp ứng viên trùng tên gần giống bằng RapidFuzz.
5. Gán tag theo taxonomy cố định. Mặc định dùng regex có bằng chứng; có thể bật Gemini để đề xuất tag, sau đó vẫn lọc lại bằng regex.
6. Xuất `tagged_entities.json` và `crawl_records.csv` để xem lại trước khi đưa vào bước embedding.

**Lưu ý:** website có thể chặn request, thay đổi HTML hoặc có nội dung tải bằng JavaScript. Không vượt CAPTCHA/đăng nhập; tôn trọng điều khoản và robots.txt của nguồn, giữ khoảng nghỉ giữa request. Kết quả crawl là dữ liệu cần kiểm duyệt, không phải danh sách địa điểm đã xác minh.

**Ảnh:** notebook chỉ lấy ảnh từ các bài viết đã crawl. Ảnh được gắn vào section gần nhất theo thứ tự DOM; heuristic loại logo/banner có thể bỏ sót hoặc loại nhầm, nên cần kiểm tra `venue_images.csv`. URL ảnh có thể bị hotlink protection hoặc thay đổi theo thời gian.


In [ ]:
%pip -q install beautifulsoup4 lxml httpx rapidfuzz pandas pyyaml

### 6.1. Khai báo nguồn crawl

Bắt đầu với một vài nguồn để kiểm tra parser; khi hoạt động ổn định mới thêm nguồn. Không chạy đồng thời quá nhiều request.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse, urljoin
import asyncio, json, re, time, unicodedata
import httpx
import pandas as pd
from bs4 import BeautifulSoup
from rapidfuzz import fuzz, process

RAW_DIR = Path("data/raw_colab")
HTML_DIR = Path("data/raw_html_colab")
PROCESSED_DIR = Path("data/processed_colab")
for p in (RAW_DIR, HTML_DIR, PROCESSED_DIR):
    p.mkdir(parents=True, exist_ok=True)

SOURCES = [
    {"name": "luhanhvietnam", "url": "https://luhanhvietnam.com.vn/du-lich/phong-tra-o-ha-noi.html"},
    {"name": "mytour_1", "url": "https://mytour.vn/vi/blog/bai-viet/10-dia-diem-phong-tra-am-nhac-tuyet-voi-o-ha-noi-de-ban-thu-gian-digifood.html"},
    {"name": "mytour_2", "url": "https://mytour.vn/vi/blog/bai-viet/thu-gian-voi-am-nhac-tai-13-dia-chi-phong-tra-ha-noi-doc-dao-va-noi-tieng-nhat.html"},
    {"name": "vinwonders", "url": "https://vinwonders.com/vi/wonderpedia/news/phong-tra-ha-noi/"},
    {"name": "digiticket", "url": "https://digiticket.vn/blog/phong-tra-ha-noi"},
    {"name": "hanoireview", "url": "https://hanoireview.vn/topAZ/phong-tra-tai-ha-noi-noi-tieng"},
]

# Có thể đặt thành SOURCES[:2] để chạy thử 2 nguồn đầu tiên trước.
MAX_SOURCES = 3
SLEEP_SECONDS = 2.0
REQUEST_TIMEOUT = 25.0
USER_AGENT = "Mozilla/5.0 (compatible; EducationalResearchBot/1.0)"
print(f"Sẽ thử crawl {min(MAX_SOURCES, len(SOURCES))}/{len(SOURCES)} nguồn")

### 6.2. Crawl HTML và lưu bản thô

Mỗi nguồn được xử lý tuần tự để giảm tải. HTML gốc được lưu riêng để có thể sửa parser mà không phải tải lại trang.

In [ ]:
async def crawl_one(client, source):
    name, url = source["name"], source["url"]
    try:
        response = await client.get(url, follow_redirects=True)
        response.raise_for_status()
        html = response.text
        (HTML_DIR / f"{name}.html").write_text(html, encoding="utf-8")
        record = {
            "source": name, "url": url, "status_code": response.status_code,
            "html_file": str(HTML_DIR / f"{name}.html"), "html_chars": len(html),
            "fetched_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        print(f"OK {name}: HTTP {response.status_code}, {len(html):,} chars")
        return {**record, "html": html}
    except Exception as e:
        print(f"FAIL {name}: {type(e).__name__}: {e}")
        return {"source": name, "url": url, "status": "failed", "error": str(e), "html": ""}

async def crawl_sources(sources):
    headers = {"User-Agent": USER_AGENT, "Accept-Language": "vi,en-US;q=0.8,en;q=0.7"}
    results = []
    async with httpx.AsyncClient(headers=headers, timeout=REQUEST_TIMEOUT) as client:
        for source in sources:
            results.append(await crawl_one(client, source))
            await asyncio.sleep(SLEEP_SECONDS)
    return results

crawl_results = await crawl_sources(SOURCES[:MAX_SOURCES])
for item in crawl_results:
    item.pop("html", None)
(RAW_DIR / "crawl_summary.json").write_text(
    json.dumps(crawl_results, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.DataFrame(crawl_results)[[c for c in ["source", "url", "status_code", "html_chars", "status", "error"] if c in pd.DataFrame(crawl_results).columns]]

### 6.3. Clean HTML và trích xuất section

Parser chung dưới đây chỉ là baseline. Nó xóa các thẻ giao diện thường gặp và thu thập nội dung theo `h2/h3` cùng các đoạn văn kế tiếp. Một số website có cấu trúc riêng nên có thể cần parser riêng giống `BlogListCrawler` trong repository.

In [ ]:
BOILERPLATE_SELECTORS = [
    "script", "style", "noscript", "svg", "iframe", "nav", "footer", "aside",
    "form", "button", "[role='navigation']", ".breadcrumb", ".share", ".social",
    ".advertisement", ".ads", ".related-posts", ".comments", ".site-footer", ".site-header",
]
BOILERPLATE_TEXT = re.compile(
    r"^(đăng nhập|đăng ký|chia sẻ bài viết|bài viết liên quan|tin liên quan|xem thêm|quảng cáo)$",
    re.I,
)

# FIX 1: dùng ranh giới từ. Regex cũ "ads?" khớp cả "uploads", "download", "header"...
# nên gần như mọi ảnh trong wp-content/uploads bị loại nhầm.
BAD_IMAGE_HINT = re.compile(
    r"(?<![a-z0-9])(logo|icon|icons|avatar|sprite|banner|advert|ad|ads|favicon|"
    r"tracking|pixel|loading|placeholder|spacer|blank|emoji)(?![a-z0-9])"
)
IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".gif", ".avif", ".bmp")

def clean_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return re.sub(r"\s+", " ", text).strip()

def _best_from_srcset(srcset):
    """FIX 2: chọn ảnh có width lớn nhất; không split theo dấu phẩy trần (URL CDN có thể chứa dấu phẩy)."""
    best_url, best_w = None, -1
    for part in re.split(r",\s+", srcset.strip()):
        bits = part.strip().split()
        if not bits:
            continue
        url = bits[0]
        w = 0
        if len(bits) > 1:
            m = re.match(r"(\d+)(w|x)", bits[1])
            if m:
                w = int(m.group(1)) * (1000 if m.group(2) == "x" else 1)
        if w >= best_w:
            best_url, best_w = url, w
    return best_url

def _dim(value):
    """FIX 3: width="100%" từng bị đọc thành 100 (<120) và loại ảnh. Bỏ qua giá trị phần trăm/không phải px."""
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s or "%" in s or s in ("auto", "inherit"):
        return None
    m = re.match(r"^(\d+)(px)?$", s)
    return int(m.group(1)) if m else None

def _img_candidate(img, page_url):
    """Lấy URL ảnh từ thuộc tính HTML phổ biến (kể cả lazy-load, <picture>); lọc nhanh logo/icon/banner rõ ràng."""
    raw = None
    for attr in ("data-src", "data-lazy-src", "data-original", "data-lazy", "data-full-url", "src"):
        v = img.get(attr)
        if v and not v.strip().startswith(("data:", "blob:")):
            raw = v.strip()
            break
    if not raw:
        srcset = img.get("data-srcset") or img.get("srcset")
        if not srcset:
            pic = img.find_parent("picture")  # lxml đôi khi lồng <img> vào trong <source>, nên tìm theo tổ tiên
            if pic is not None:
                for src_tag in pic.find_all("source"):
                    srcset = src_tag.get("data-srcset") or src_tag.get("srcset")
                    if srcset:
                        break
        if srcset:
            raw = _best_from_srcset(srcset)
    if not raw or raw.startswith(("data:", "blob:")):
        return None
    url = urljoin(page_url, raw)
    parsed = urlparse(url)
    if parsed.scheme not in ("http", "https") or not parsed.netloc:
        return None
    cls = img.get("class", "")
    cls = " ".join(cls) if isinstance(cls, list) else (cls or "")
    # Chỉ xét tên file + alt/class/id; KHÔNG xét cả thư mục (uploads, ads-cdn...) để tránh loại nhầm.
    filename = parsed.path.rsplit("/", 1)[-1]
    hint = " ".join([img.get("alt", "") or "", img.get("title", "") or "", cls,
                     img.get("id", "") or "", filename]).casefold()
    if BAD_IMAGE_HINT.search(hint):
        return None
    ext = "." + filename.rsplit(".", 1)[-1].lower() if "." in filename else ""
    if ext in (".svg", ".ico"):
        return None
    w, h = _dim(img.get("width")), _dim(img.get("height"))
    if w is not None and w < 120:
        return None
    if h is not None and h < 80:
        return None
    return url

def parse_article_sections(html, source_name, url):
    soup = BeautifulSoup(html, "lxml")
    page_title = clean_text(soup.title.get_text(" ", strip=True)) if soup.title else ""
    for selector in BOILERPLATE_SELECTORS:
        for node in soup.select(selector): node.decompose()
    root = soup.find("article") or soup.find("main") or soup.body or soup
    sections, current_heading, current_paras, current_images = [], page_title, [], []
    def flush_section():
        # FIX 4: section chỉ có ảnh (không có đoạn văn) cũng được giữ lại ảnh, không bị bỏ như trước.
        if current_paras or current_images:
            sections.append({"source": source_name, "url": url, "page_title": page_title,
                             "heading": current_heading, "text": " ".join(current_paras)[:5000],
                             "image_urls": list(dict.fromkeys(current_images))})
    # Duyệt theo thứ tự DOM để gắn ảnh vào section gần nhất, không gom ảnh toàn trang vào mọi quán.
    for node in root.find_all(["h1", "h2", "h3", "h4", "p", "li", "img"]):
        if node.name == "img":
            image_url = _img_candidate(node, url)
            if image_url: current_images.append(image_url)
            continue
        text = clean_text(node.get_text(" ", strip=True))
        if not text or BOILERPLATE_TEXT.match(text) or len(text) < 3: continue
        if node.name in ("h1", "h2", "h3", "h4"):
            flush_section()
            current_heading, current_paras, current_images = text, [], []
        elif len(text) >= 25:
            current_paras.append(text)
    flush_section()
    # Section chỉ có ảnh (text rỗng) bị loại ở bước này để không tạo entity rỗng,
    # nhưng ảnh của nó đã được gộp vào section kế tiếp qua flush ở trên nếu cùng heading.
    return [s for s in sections if len(s["text"]) >= 40]

html_by_source = {}
for source in SOURCES[:MAX_SOURCES]:
    p = HTML_DIR / f"{source['name']}.html"
    if p.exists(): html_by_source[source["name"]] = (source["url"], p.read_text(encoding="utf-8", errors="ignore"))

sections = []
for source_name, (url, html) in html_by_source.items():
    parsed = parse_article_sections(html, source_name, url)
    print(f"{source_name}: {len(parsed)} sections; {sum(len(s.get('image_urls', [])) for s in parsed)} image candidates")
    sections.extend(parsed)
(RAW_DIR / "sections.json").write_text(json.dumps(sections, ensure_ascii=False, indent=2), encoding="utf-8")
print("Tổng sections:", len(sections))
pd.DataFrame(sections).head(10)


### 6.4. Trích xuất ứng viên địa điểm

Trong bài dạng danh sách, tên quán thường nằm ở heading. Bước này dùng heuristic để tạo ứng viên, giữ lại URL nguồn và đoạn mô tả để có thể truy vết. Không nên coi mọi heading là một địa điểm; hãy xem bảng review trước khi tiếp tục.

In [ ]:
LISTICLE_PREFIX = re.compile(r"^\s*(?:(?:địa điểm|quán|phòng trà)?\s*)?(?:\d{1,2}[.)]|#\d{1,2}\b)\s*[-.:)]*\s*", re.I)
GENERIC_HEADINGS = {
    "lời kết", "kết luận", "mở đầu", "giới thiệu", "thông tin liên hệ", "câu hỏi thường gặp",
    "phòng trà hà nội", "top phòng trà hà nội", "mục lục", "xem thêm"
}
def candidate_name(heading):
    name = LISTICLE_PREFIX.sub("", clean_text(heading)).strip(" -:|·")
    name = re.sub(r"\s+[–—-]\s+.*$", "", name).strip()
    if len(name) < 3 or len(name) > 90 or name.lower() in GENERIC_HEADINGS: return ""
    if any(x in name.lower() for x in ["những phòng trà", "phòng trà nổi tiếng nhất", "địa điểm phòng trà"]): return ""
    return name

candidates = []
for sec in sections:
    name = candidate_name(sec["heading"])
    if not name: continue
    candidates.append({
        "name": name, "aliases": [],
        "descriptions": [{"source": sec["source"], "url": sec["url"], "text": sec["text"]}],
        "source_urls": [sec["url"]], "source_names": [sec["source"]],
        "image_candidates": [{"url": u, "source_url": sec["url"], "source_name": sec["source"]} for u in sec.get("image_urls", [])],
    })
print("Ứng viên thô:", len(candidates))
pd.DataFrame([{"name":x["name"], "source":x["source_names"][0], "image_candidates":len(x["image_candidates"]), "text_preview":x["descriptions"][0]["text"][:180]} for x in candidates]).head(30)


### 6.5. Gộp ứng viên trùng lặp

Dùng RapidFuzz để gợi ý tên gần giống. Ngưỡng cao giúp hạn chế gộp nhầm nhưng vẫn có thể bỏ sót tên viết khác nhau; với dữ liệu quan trọng, cần rà soát thủ công. Đây là bản rút gọn, không thay thế hoàn toàn `src/processing/entity_resolver.py`.

In [ ]:
from copy import deepcopy
FUZZY_THRESHOLD = 90
merged_entities = []
for item in candidates:
    norm = clean_text(item["name"]).casefold()
    match_idx, match_score = None, 0
    for idx, existing in enumerate(merged_entities):
        score = fuzz.token_set_ratio(norm, clean_text(existing["name"]).casefold())
        if score > match_score: match_idx, match_score = idx, score
    if match_idx is not None and match_score >= FUZZY_THRESHOLD:
        target = merged_entities[match_idx]
        if item["name"] != target["name"] and item["name"] not in target["aliases"]: target["aliases"].append(item["name"])
        target["descriptions"].extend(item["descriptions"])
        target["source_urls"] = list(dict.fromkeys(target["source_urls"] + item["source_urls"]))
        target["source_names"] = list(dict.fromkeys(target["source_names"] + item["source_names"]))
        target["image_candidates"].extend(item.get("image_candidates", []))
    else:
        merged_entities.append(deepcopy(item))
for i, entity in enumerate(merged_entities, 1):
    entity["id"] = f"venue-{i:04d}"
    seen, unique_desc = set(), []
    for d in entity["descriptions"]:
        key = clean_text(d["text"]).casefold()
        if key and key not in seen: seen.add(key); unique_desc.append(d)
    entity["descriptions"] = unique_desc
    # Deduplicate image URLs while retaining source metadata.
    seen_urls, unique_images = set(), []
    for img in entity.get("image_candidates", []):
        u = img.get("url", "")
        if u and u not in seen_urls:
            seen_urls.add(u); unique_images.append(img)
    entity["image_candidates"] = unique_images
print(f"Ứng viên trước gộp: {len(candidates)} | sau gộp: {len(merged_entities)}")
pd.DataFrame([{"id":e["id"], "name":e["name"], "aliases":e["aliases"], "n_descriptions":len(e["descriptions"]), "image_candidates":len(e.get("image_candidates", [])), "sources":e["source_names"]} for e in merged_entities]).head(30)


### 6.5.1. Thu thập, kiểm tra và gắn ảnh cho từng địa điểm

Ảnh được lấy **chỉ từ các bài viết đã crawl**, không dùng Google Places. Parser gắn ảnh vào section gần nhất trong DOM, sau đó ứng viên ảnh được gộp cùng entity. Cell dưới kiểm tra URL bằng GET stream, xác nhận content-type là ảnh, lọc tên/logo/banner bằng heuristic, và giữ tối đa 5 ảnh hợp lệ mỗi địa điểm (mục tiêu 3–5 nếu nguồn có đủ). Đây không phải nhận diện ảnh bằng thị giác; cần xem lại ảnh và nguồn thủ công.

**Các lỗi đã sửa trong phiên bản này:** (1) regex lọc ảnh `ads?` khớp nhầm `uploads`/`download`/`header` nên loại gần hết ảnh; (2) `width="100%"` bị đọc thành 100px và bị loại; (3) không gửi `Referer` nên bị chặn hotlink (403); (4) Range request bị từ chối không có phương án dự phòng; (5) server trả `octet-stream` bị coi là không phải ảnh; (6) section chỉ có ảnh bị bỏ; (7) thêm lọc ảnh dùng chung nhiều quán (sidebar/quảng cáo) và hỗ trợ `<picture>`/`srcset`.


In [ ]:
# Kiểm tra URL ảnh có phản hồi và là ảnh thật.
# FIX 5: gửi Referer = trang bài viết gốc (nhiều site chặn hotlink nếu thiếu Referer -> 403).
# FIX 6: nếu Range bị từ chối (400/403/405/416) thì thử lại không Range.
# FIX 7: chấp nhận Content-Type octet-stream nếu đuôi file là ảnh; kiểm tra thêm magic bytes.
import mimetypes
from urllib.parse import urlparse

MAX_IMAGES_PER_VENUE = 5
MIN_TARGET_IMAGES = 3
IMAGE_CHECK_TIMEOUT = 15.0
IMAGE_CHECK_PAUSE = 0.15
SHARED_IMAGE_MAX_VENUES = 3   # ảnh xuất hiện ở >= số này quán khác nhau thường là widget/sidebar -> bỏ

def _looks_like_image(head: bytes) -> bool:
    return (head.startswith(b"\xff\xd8\xff") or head.startswith(b"\x89PNG") or head.startswith(b"GIF8")
            or (head[:4] == b"RIFF" and head[8:12] == b"WEBP") or head[4:12] in (b"ftypavif", b"ftypavis"))

async def _try_image_get(client, url, headers):
    async with client.stream("GET", url, headers=headers, follow_redirects=True,
                             timeout=IMAGE_CHECK_TIMEOUT) as response:
        status = response.status_code
        if status >= 400:
            return None, f"http_{status}", None
        ctype = response.headers.get("content-type", "").split(";")[0].lower().strip()
        head = b""
        async for chunk in response.aiter_bytes():
            head += chunk
            if len(head) >= 16:
                break
        ext = mimetypes.guess_type(urlparse(str(response.url)).path)[0] or ""
        if ctype.startswith("image/") and "svg" not in ctype:
            ok = True
        elif ctype in ("application/octet-stream", "binary/octet-stream", "") and (ext.startswith("image/") or _looks_like_image(head)):
            ok = True
            ctype = ext or "image/unknown"
        else:
            ok = False
        if not ok:
            return None, f"not_image_content_type:{ctype or 'missing'}", None
        return {"url": str(response.url), "content_type": ctype, "status_code": status}, "ok", None

async def validate_image_url(client, image_record):
    url = image_record.get("url", "")
    referer = image_record.get("source_url") or ""
    base_headers = {"Referer": referer} if referer else {}
    attempts = [{**base_headers, "Range": "bytes=0-2047"}, dict(base_headers)]
    last_reason = "unknown"
    for headers in attempts:
        try:
            info, reason, _ = await _try_image_get(client, url, headers)
            if info:
                return {
                    "url": info["url"],
                    "source_url": referer,
                    "source_name": image_record.get("source_name", ""),
                    "content_type": info["content_type"],
                    "status_code": info["status_code"],
                    "alt": image_record.get("alt", ""),
                }, "ok"
            last_reason = reason
            # Chỉ thử lại khi lỗi có thể do Range/hotlink; lỗi 404/410 thì dừng.
            if not (reason.startswith("http_") and reason.split("_")[1] in ("400", "403", "405", "416")):
                break
        except Exception as exc:
            last_reason = f"{type(exc).__name__}:{str(exc)[:120]}"
    return None, last_reason

def drop_shared_images(entities):
    """Bỏ ảnh xuất hiện ở quá nhiều quán (thường là ảnh sidebar/quảng cáo, không phải ảnh quán)."""
    counts = {}
    for e in entities:
        for u in {c.get("url") for c in e.get("image_candidates", []) if c.get("url")}:
            counts[u] = counts.get(u, 0) + 1
    removed = 0
    for e in entities:
        kept = [c for c in e.get("image_candidates", []) if counts.get(c.get("url"), 0) < SHARED_IMAGE_MAX_VENUES]
        removed += len(e.get("image_candidates", [])) - len(kept)
        e["image_candidates"] = kept
    print(f"Đã bỏ {removed} ảnh dùng chung ở >= {SHARED_IMAGE_MAX_VENUES} quán")

async def validate_entity_images(entities):
    drop_shared_images(entities)
    headers = {"User-Agent": USER_AGENT, "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
               "Accept-Language": "vi,en-US;q=0.8,en;q=0.7"}
    async with httpx.AsyncClient(headers=headers, timeout=IMAGE_CHECK_TIMEOUT, follow_redirects=True) as client:
        for idx, entity in enumerate(entities, 1):
            valid, errors, seen = [], [], set()
            for candidate in entity.get("image_candidates", []):
                url = candidate.get("url", "")
                if not url or url in seen: continue
                seen.add(url)
                checked, status = await validate_image_url(client, candidate)
                if checked:
                    valid.append(checked)
                else:
                    errors.append({"url": url, "reason": status, "source_url": candidate.get("source_url", "")})
                if len(valid) >= MAX_IMAGES_PER_VENUE: break
                await asyncio.sleep(IMAGE_CHECK_PAUSE)
            entity["images"] = valid
            entity["image_urls"] = [x["url"] for x in valid]
            entity["image_url"] = valid[0]["url"] if valid else ""
            entity["image_status"] = "ok" if len(valid) >= MIN_TARGET_IMAGES else ("partial" if valid else "missing")
            entity["image_check_errors"] = errors[:20]
            reasons = ", ".join(sorted({e["reason"].split(":")[0] for e in errors})) or "-"
            print(f"[{idx}/{len(entities)}] {entity['name']}: {len(valid)} ảnh hợp lệ ({entity['image_status']}); "
                  f"candidate={len(entity.get('image_candidates', []))}; lỗi: {reasons}")
            await asyncio.sleep(0.1)
    return entities

merged_entities = await validate_entity_images(merged_entities)
image_summary = pd.DataFrame([{
    "name": e["name"], "image_count": len(e.get("images", [])),
    "image_status": e.get("image_status"), "image_url": e.get("image_url", ""),
    "sources": ",".join(sorted(set(x.get("source_name", "") for x in e.get("images", []))))
} for e in merged_entities])
display(image_summary)
print("Ảnh mục tiêu: 3–5 mỗi địa điểm khi nguồn có đủ. 'partial'/'missing' cần xem lại thủ công hoặc thêm nguồn crawl phù hợp.")


### 6.6. Gán tag theo taxonomy + kiểm chứng bằng bằng chứng

Để chạy được ngay không cần API key, mặc định lab đề xuất tag bằng regex trên mô tả rồi chỉ giữ tag có bằng chứng. Cách này dễ giải thích nhưng kém linh hoạt hơn LLM. Có thể bật Gemini ở cell kế tiếp để đề xuất tag; ngay cả khi dùng LLM, kết quả vẫn phải qua danh sách tag cho phép và bước kiểm chứng bằng regex.

In [ ]:
TAG_TAXONOMY = {
    "music_genres": {
        "nhac_trinh": r"nhac trinh|trinh cong son|nhac si trinh",
        "acoustic": r"acoustic|guitar",
        "jazz": r"\bjazz\b",
        "bolero": r"bolero|nhac vang|tien chien",
        "lofi_chill": r"lofi|lo-fi|chill",
        "indie": r"\bindie\b|underground",
        "traditional": r"dan gian|co truyen|dan toc|dan ca|ca tru|cheo|quan ho",
        "piano": r"\bpiano\b",
        "pop_ballad": r"\bpop\b|ballad|nhac tre|nhac tinh cam",
    },
    "atmosphere_tags": {
        "yen_tinh": r"yen tinh|yen binh|an yen|thanh tinh|nhe nhang|thu gian|thu thai|peaceful|quiet",
        "lang_man": r"lang man|hen ho|cap doi|romantic",
        "co_kinh": r"co kinh|hoai co|hoai niem|co xua|nha co|retro|vintage|classic",
        "view_dep": r"\bview\b|toan canh|nhin ra|ho tay|canh quan",
        "san_vuon": r"san vuon|khu vuon|vuon cay|khong gian xanh|garden|outdoor",
        "rooftop": r"san thuong|tang thuong|tang mai|rooftop",
        "sang_trong": r"sang trong|cao cap|dang cap|xa hoa|luxury|elegant",
        "binh_dan": r"binh dan|gia re|phai chang|sinh vien|affordable|budget",
        "tre_trung": r"tre trung|hien dai|nang dong|youth|modern|trendy",
    },
    "features": {
        "co_live_music": r"nhac song|\blive\b|bieu dien|trinh dien|ca si|ban nhac|san khau|dem nhac|hat truc tiep|nghe si",
        "cho_dat_cho": r"dat cho|dat ban|reservation|\bbook\b",
        "do_an_nhe": r"do an nhe|banh|snack|mon an|thuc don|food",
        "parking": r"gui xe|do xe|bai xe|bai do|parking|cho de xe",
        "khong_thuoc_la": r"khong hut thuoc|cam hut thuoc|non-smoking|no smoking",
        "wifi_manh": r"wi-?fi",
    },
}

def strip_accents(text):
    text = unicodedata.normalize("NFD", text or "")
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("đ", "d").replace("Đ", "D").lower()

def get_entity_text(entity):
    return " ".join(d.get("text", "") for d in entity.get("descriptions", []))[:4000]

def rule_based_tags(entity):
    text = strip_accents(get_entity_text(entity))
    names = [entity.get("name", "")] + entity.get("aliases", [])
    music_text = text
    for name in names:
        n = strip_accents(name).strip()
        if len(n) >= 4:
            music_text = music_text.replace(n, " ")
    result = {group: [] for group in TAG_TAXONOMY}
    for group, tags in TAG_TAXONOMY.items():
        haystack = music_text if group == "music_genres" else text
        for tag, pattern in tags.items():
            if re.search(pattern, haystack, flags=re.I):
                result[group].append(tag)
    return result

for entity in merged_entities:
    entity.update(rule_based_tags(entity))
    entity["tagging_method"] = "regex_evidence"

print("Đã gán tag cho", len(merged_entities), "entity")
pd.DataFrame([{"name":e["name"], "music_genres":e["music_genres"], "atmosphere_tags":e["atmosphere_tags"], "features":e["features"]} for e in merged_entities]).head(30)

### 6.7. (Tùy chọn) Gemini đề xuất tag, sau đó kiểm chứng

Bỏ qua cell này nếu chưa có API key. Trong Colab, đặt `GEMINI_API_KEY` qua **Secrets** (biểu tượng chìa khóa), không ghi key trực tiếp vào notebook hoặc đẩy lên GitHub. API có thể tính phí hoặc giới hạn quota tùy tài khoản/model.

Cell này chỉ minh họa cách gọi Gemini qua `google-genai`; nó không tuyên bố là bản sao nguyên vẹn của `src/processing/llm_tagger.py`. Kết quả LLM được lọc theo taxonomy và chỉ giữ tag có bằng chứng trong text.

In [ ]:
# Tùy chọn: đổi thành True sau khi đã cấu hình Colab Secret GEMINI_API_KEY.
USE_GEMINI_TAGGING = False

if USE_GEMINI_TAGGING:
    %pip -q install google-genai
    from google import genai
    from google.genai import types
    from getpass import getpass
    import os
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        api_key = None
    if not api_key:
        api_key = getpass("Nhập GEMINI_API_KEY (không hiển thị): ")
    client = genai.Client(api_key=api_key)

    taxonomy_text = json.dumps({g: list(v.keys()) for g, v in TAG_TAXONOMY.items()}, ensure_ascii=False)
    for i, entity in enumerate(merged_entities):
        text = get_entity_text(entity)
        if len(text) < 30:
            continue
        prompt = f"""Bạn phân loại dữ liệu phòng trà Hà Nội. Chỉ trả JSON hợp lệ, không markdown.
Chỉ dùng tag có trong taxonomy: {taxonomy_text}
Không suy đoán nếu mô tả không có bằng chứng. Mỗi nhóm tối đa 4 tag.
Tên địa điểm: {entity['name']}
Mô tả nguồn: {text[:3500]}
Schema: {{\"music_genres\":[],\"atmosphere_tags\":[],\"features\":[]}}"""
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
                config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0),
            )
            proposed = json.loads(response.text)
            # Chỉ giữ tag nằm trong taxonomy VÀ có bằng chứng regex trong mô tả.
            evidence = rule_based_tags(entity)
            for group in TAG_TAXONOMY:
                allowed = set(TAG_TAXONOMY[group])
                requested = proposed.get(group, []) if isinstance(proposed, dict) else []
                entity[group] = [t for t in dict.fromkeys(requested) if t in allowed and t in evidence[group]]
            entity["tagging_method"] = "gemini_proposal_plus_regex_validation"
        except Exception as exc:
            print(f"Gemini lỗi ở entity {i} ({entity['name']}): {exc}; giữ tag regex hiện có")
        time.sleep(1.0)
    print("Hoàn tất tag tùy chọn bằng Gemini + kiểm chứng regex")
else:
    print("Đang dùng regex evidence-only; không cần API key.")

### 6.8. Xuất dữ liệu và kiểm tra thủ công

Tệp đầu ra chứa mô tả và URL nguồn để truy vết. Hãy kiểm tra tên quán, mô tả và tag trước khi sử dụng trong hệ thống. Địa chỉ, giá và giờ mở cửa không được tự suy ra nếu nguồn không nêu rõ; ở lab này các trường đó để trống.

In [ ]:
# Chỉ giữ entity có tên và ít nhất một mô tả có nội dung.
tagged_entities = []
for entity in merged_entities:
    entity["descriptions"] = [d for d in entity.get("descriptions", []) if clean_text(d.get("text", ""))]
    if not entity.get("name") or not entity["descriptions"]: continue
    entity.setdefault("address", "")
    entity.setdefault("district", "")
    entity.setdefault("price_min", None)
    entity.setdefault("price_max", None)
    entity.setdefault("hours", "")
    entity.setdefault("images", [])
    entity.setdefault("image_urls", [x.get("url", "") for x in entity["images"] if x.get("url")])
    entity.setdefault("image_url", entity["image_urls"][0] if entity["image_urls"] else "")
    entity.setdefault("image_status", "ok" if len(entity["image_urls"]) >= 3 else ("partial" if entity["image_urls"] else "missing"))
    tagged_entities.append(entity)

json_path = PROCESSED_DIR / "tagged_entities.json"
csv_path = PROCESSED_DIR / "crawl_records.csv"
image_csv_path = PROCESSED_DIR / "venue_images.csv"
json_path.write_text(json.dumps(tagged_entities, ensure_ascii=False, indent=2), encoding="utf-8")
flat_rows, image_rows = [], []
for e in tagged_entities:
    for d in e["descriptions"]:
        flat_rows.append({
            "id": e["id"], "name": e["name"], "source": d.get("source"), "url": d.get("url"),
            "description": d.get("text"), "music_genres": ",".join(e["music_genres"]),
            "atmosphere_tags": ",".join(e["atmosphere_tags"]), "features": ",".join(e["features"]),
            "image_url": e.get("image_url", ""), "image_count": len(e.get("images", [])),
            "image_status": e.get("image_status", "missing"),
        })
    for rank, img in enumerate(e.get("images", []), 1):
        image_rows.append({
            "venue_id": e["id"], "venue_name": e["name"], "image_rank": rank,
            "image_url": img.get("url", ""), "source_name": img.get("source_name", ""),
            "source_url": img.get("source_url", ""), "content_type": img.get("content_type", ""),
            "status_code": img.get("status_code", ""), "alt": img.get("alt", ""),
        })
pd.DataFrame(flat_rows).to_csv(csv_path, index=False, encoding="utf-8-sig")
pd.DataFrame(image_rows, columns=["venue_id", "venue_name", "image_rank", "image_url", "source_name", "source_url", "content_type", "status_code", "alt"]).to_csv(image_csv_path, index=False, encoding="utf-8-sig")
print("Entity sau xử lý:", len(tagged_entities))
print("JSON:", json_path)
print("CSV bản ghi mô tả:", csv_path)
print("CSV metadata ảnh:", image_csv_path)
print("Tổng ảnh hợp lệ:", sum(len(e.get("images", [])) for e in tagged_entities))
print("Số entity có 3+ ảnh:", sum(len(e.get("images", [])) >= 3 for e in tagged_entities))
print("Nếu số entity = 0: kiểm tra lỗi crawl, cấu trúc heading hoặc chạy ít nguồn hơn để debug.")


### 6.9. Đưa dữ liệu crawl vào phần embedding/search

Cell dữ liệu ở mục kế tiếp sẽ ưu tiên `tagged_entities` nếu pipeline crawl tạo ra kết quả. Nếu crawl không lấy được nội dung hoặc không có entity hợp lệ, notebook tự dùng dữ liệu giả lập để các phần embedding và search vẫn có thể chạy. Khi dùng dữ liệu crawl, kết quả tìm kiếm sẽ phụ thuộc chất lượng parser và cần review thủ công.

## 7. Dữ liệu đầu vào cho embedding

Nếu bước crawl → clean → extract → deduplicate → tag phía trên tạo được `tagged_entities`, notebook dùng dữ liệu đó. Nếu không crawl được hoặc muốn thử phần search độc lập, notebook sẽ fallback về dữ liệu giả lập (có nhãn demo, không phải thông tin thật).

In [ ]:
# Ưu tiên dữ liệu crawl thật đã qua bước tag; fallback demo khi không có dữ liệu.
if "tagged_entities" in globals() and tagged_entities:
    venues = []
    for e in tagged_entities:
        venues.append({
            "id": e["id"], "name": e["name"],
            "district": e.get("district", ""), "address": e.get("address", ""),
            "price_min": e.get("price_min"), "price_max": e.get("price_max"),
            "hours": e.get("hours", ""),
            "music_genres": e.get("music_genres", []),
            "atmosphere_tags": e.get("atmosphere_tags", []), "features": e.get("features", []),
            "descriptions": [{"source": d.get("source", "unknown"), "text": d.get("text", "")} for d in e.get("descriptions", []) if d.get("text")],
        })
    print(f"Đang dùng dữ liệu crawl: {len(venues)} quán/entity")
else:
    print("Không có dữ liệu crawl hợp lệ — dùng dataset giả lập để minh họa search.")
    # Dữ liệu giả lập phục vụ demo — không phải thông tin đã xác minh về quán thật.
    venues = [
        {
            "id": "demo-trinh-ca",
            "name": "Phòng trà Trịnh Ca (demo)",
            "district": "Ba Đình",
            "address": "Địa chỉ minh họa, Ba Đình",
            "price_min": 50000,
            "price_max": 150000,
            "hours": "19:00-23:00",
            "music_genres": ["nhac_trinh", "acoustic"],
            "atmosphere_tags": ["yen_tinh", "lang_man"],
            "features": ["co_live_music"],
            "descriptions": [
                {"source": "demo_blog_a", "text": "Không gian nghe nhạc Trịnh, acoustic với âm lượng vừa phải và chỗ ngồi thư giãn."},
                {"source": "demo_blog_b", "text": "Buổi tối phù hợp người thích ca khúc Trịnh Công Sơn và trò chuyện yên tĩnh."}
            ]
        },
        {
            "id": "demo-jazz",
            "name": "Jazz Corner (demo)",
            "district": "Hoàn Kiếm",
            "address": "Địa chỉ minh họa, Hoàn Kiếm",
            "price_min": 100000,
            "price_max": 250000,
            "hours": "20:00-00:00",
            "music_genres": ["jazz", "piano"],
            "atmosphere_tags": ["sang_trong"],
            "features": ["co_live_music"],
            "descriptions": [
                {"source": "demo_blog_a", "text": "Đêm nhạc jazz và piano trong không gian nhỏ, ánh sáng dịu."},
                {"source": "demo_blog_c", "text": "Chương trình biểu diễn jazz dành cho người yêu nhạc cụ và ứng tấu."}
            ]
        },
        {
            "id": "demo-acoustic", "name": "Acoustic Garden (demo)",
            "district": "Cầu Giấy", "address": "Địa chỉ minh họa, Cầu Giấy",
            "price_min": 40000, "price_max": 120000, "hours": "18:00-23:00",
            "music_genres": ["acoustic", "indie"],
            "atmosphere_tags": ["san_vuon", "tre_trung"],
            "features": ["co_live_music", "parking"],
            "descriptions": [
                {"source": "demo_blog_b", "text": "Sân vườn có nhạc acoustic và indie, phù hợp nhóm bạn trẻ."}
            ]
        },
        {
            "id": "demo-bolero", "name": "Bolero Mộc (demo)",
            "district": "Đống Đa", "address": "Địa chỉ minh họa, Đống Đa",
            "price_min": 60000, "price_max": 180000, "hours": "19:30-23:00",
            "music_genres": ["bolero", "nhac_trinh"],
            "atmosphere_tags": ["co_kinh", "yen_tinh"],
            "features": ["co_live_music"],
            "descriptions": [
                {"source": "demo_blog_a", "text": "Đêm nhạc bolero và tình khúc xưa trong không gian hoài niệm."}
            ]
        },
        {
            "id": "demo-rooftop", "name": "Rooftop Piano (demo)",
            "district": "Tây Hồ", "address": "Địa chỉ minh họa, Tây Hồ",
            "price_min": 120000, "price_max": 300000, "hours": "18:00-00:00",
            "music_genres": ["piano", "pop_ballad"], "atmosphere_tags": ["rooftop", "view_dep"], "features": ["co_live_music", "cho_dat_cho"],
            "descriptions": [
                {"source": "demo_blog_c", "text": "Piano và pop ballad trên tầng thượng, có tầm nhìn rộng vào buổi tối."}
            ]
        },
        {
            "id": "demo-lofi",
            "name": "Lofi Chill Room (demo)",
            "district": "Hai Bà Trưng",
            "address": "Địa chỉ minh họa, Hai Bà Trưng",
            "price_min": 30000,
            "price_max": 90000,
            "hours": "17:00-22:00",
            "music_genres": ["lofi_chill", "indie"],
            "atmosphere_tags": ["yen_tinh", "binh_dan"],

            "features": ["wifi_manh"],
            "descriptions": [
                {"source": "demo_blog_b", "text": "Không gian bình dân, nhạc lofi chill nhẹ nhàng để thư giãn sau giờ học."}
            ]
        }
    ]

print("Số quán đầu vào:", len(venues))
print("Số mô tả (sẽ trở thành số point):", sum(len(v["descriptions"]) for v in venues))
pd.DataFrame([{"id": v["id"], "name": v["name"], "district": v["district"], "description_count": len(v["descriptions"]), "tags": v["music_genres"]} for v in venues])


## 8. Gán tag


### 8.1. chiến lược gán tag
rong src/processing/llm_tagger.py của dự án Semantic Search phòng trà Hà Nội, bạn đang sử dụng chiến lược gán nhãn kết hợp LLM với bộ luật kiểm chứng bằng Regex (LLM-based tagging + rule-based validation).

Điểm quan trọng là: LLM đề xuất tag, nhưng code Python kiểm tra lại bằng luật và loại bỏ tag không có bằng chứng trong mô tả. Đây không phải kiểu để LLM tự do gán nhãn rồi lưu kết quả ngay.



+ Dữ liệu đã làm sạch: Tên quán, bí danh, mô tả từ nhiều nguồn

-> LLM phân loại theo taxonomy: Gemini trả về JSON gồm các nhóm tag

-> clean_tags():Chỉ giữ tag thuộc danh sách cho phép, loại trùng

-> validate_tags(): Chuẩn hóa alias và kiểm chứng bằng Regex

-> Lưu tag đã được kiểm chứng: music_genres · atmosphere_tags · features

### 8.2. Bộ luật gán nhãn gồm những gì?
# Chiến lược gán nhãn trong `src/processing/llm_tagger.py`

Trong `src/processing/llm_tagger.py` của dự án Semantic Search phòng trà Hà Nội, chiến lược gán nhãn được áp dụng là sự kết hợp giữa **LLM với bộ luật kiểm chứng bằng Regex** (*LLM-based tagging + rule-based validation*).

Điểm quan trọng: LLM đề xuất tag, nhưng code Python kiểm tra lại bằng luật và loại bỏ tag không có bằng chứng trong mô tả. Đây không phải kiểu để LLM tự do gán nhãn rồi lưu kết quả ngay.

---

## 1. Luồng gán tag trong code

1. **Dữ liệu đã làm sạch:** Tên quán, bí danh, mô tả từ nhiều nguồn.
2. **LLM phân loại theo taxonomy:** Gemini trả về JSON gồm các nhóm tag.
3. **`clean_tags()`:** Chỉ giữ tag thuộc danh sách cho phép, loại trùng.
4. **`validate_tags()`:** Chuẩn hóa alias và kiểm chứng bằng Regex.
5. **Lưu tag đã được kiểm chứng:** `music_genres` · `atmosphere_tags` · `features`.

---

## 2. Bộ luật gán nhãn gồm những gì?

### Luật 1: Chỉ được chọn tag trong taxonomy định nghĩa sẵn

Code khai báo ba nhóm tag cố định.

| Nhóm | Ví dụ tag | Ý nghĩa |
| :--- | :--- | :--- |
| `music_genres` | `nhac_trinh`, `acoustic`, `jazz`, `bolero`, `piano` | Thể loại nhạc |
| `atmosphere_tags` | `yen_tinh`, `lang_man`, `co_kinh`, `view_dep`, `rooftop` | Không gian và bầu không khí |
| `features` | `co_live_music`, `cho_dat_cho`, `parking`, `wifi_manh` | Dịch vụ và tiện ích |

LLM được cung cấp ý nghĩa của từng tag trong `TAXONOMY_PROMPT`. Nhờ vậy, mô hình không cần tự sáng tạo tên nhãn mới.

> **Ví dụ:** Code chỉ chấp nhận `nhac_trinh`, không chấp nhận một nhãn tự tạo như `nhac_trinh_cuc_hay`.

---

### Luật 2: Chỉ gán khi có bằng chứng trong mô tả

Đây là phần đáng chú ý nhất trong thiết kế. Hàm `validate_tags()` kiểm tra từng tag bằng các biểu thức chính quy trong dictionary `EVIDENCE`.

| Tag | Bằng chứng được code tìm kiếm |
| :--- | :--- |
| `nhac_trinh` | “nhạc Trịnh”, “Trịnh Công Sơn”, “nhạc sĩ Trịnh”… |
| `acoustic` | “acoustic”, “guitar” |
| `jazz` | “jazz” |
| `yen_tinh` | “yên tĩnh”, “yên bình”, “thanh tịnh”, “peaceful”… |
| `co_live_music` | “nhạc sống”, “biểu diễn”, “ca sĩ”, “ban nhạc”, “sân khấu”… |
| `parking` | “gửi xe”, “bãi đỗ”, “chỗ để xe”, “parking”… |

Code dùng `fx.strip_accents()` để bỏ dấu tiếng Việt trước khi so khớp. Vì thế, các cách viết có dấu và không dấu có thể được đối chiếu theo cùng một mẫu.

**Ví dụ minh họa:**

Mô tả quán có câu: *“Không gian yên tĩnh, thường xuyên tổ chức đêm nhạc Trịnh.”*
* `yen_tinh`: có bằng chứng “yên tĩnh” → **Giữ**
* `nhac_trinh`: có bằng chứng “nhạc Trịnh” → **Giữ**
* `co_live_music`: có thể giữ nếu mô tả chứa cụm phù hợp như “đêm nhạc” → **Giữ**
* `parking`: không có bằng chứng → **Loại** nếu LLM đề xuất.

*Lưu ý:* Regex chỉ kiểm tra sự xuất hiện của mẫu từ khóa, không thực sự hiểu toàn bộ ngữ cảnh. Vì vậy, nó vẫn có thể bỏ sót cách diễn đạt mới hoặc giữ nhầm tag trong một số trường hợp.

---

### Luật 3: Tránh gán nhãn sai do tên quán

Code có cơ chế xử lý riêng cho thể loại nhạc:
* Hàm `_prepare()` loại tên quán và bí danh khỏi phần văn bản dùng để kiểm tra tag thể loại nhạc.
* `GENERIC_CATEGORY` xử lý trường hợp tiêu đề bài viết dùng cụm chung như “quán cafe Acoustic Hà Nội”.

Mục đích là tránh suy luận sai rằng một quán chắc chắn chơi nhạc acoustic chỉ vì từ “Acoustic” xuất hiện trong tên quán hoặc tiêu đề danh sách. Đây là một quy tắc hữu ích khi dữ liệu được crawl từ các bài viết tổng hợp nhiều địa điểm.

---

### Luật 4: Chuẩn hóa tag đồng nghĩa

Trong code khai báo:

```python
TAG_ALIASES = {"co_dien": "co_kinh"}
```

Sau khi LLM trả về kết quả, `apply_aliases()` chuyển `co_dien` thành `co_kinh` và loại các tag trùng nhau. Mục đích là giảm tình trạng cùng một ý nghĩa bị lưu dưới nhiều nhãn khác nhau.

*Lưu ý:* Hai khái niệm “cổ điển” và “cổ kính” không phải lúc nào cũng đồng nghĩa hoàn toàn. Quy tắc gộp này là một quyết định thiết kế của dự án, nên cần kiểm tra xem có làm mất thông tin về phong cách kiến trúc hay không.

---

### Luật 5: Hạn chế tag quá mức

Prompt yêu cầu LLM:
* Không đoán khi không có căn cứ.
* Không gán các tag mâu thuẫn, ví dụ `sang_trong` và `binh_dan`.
* Thường chỉ chọn 1–4 tag nổi bật cho mỗi quán.
* Trả về JSON đúng cấu trúc.

Lưu ý rằng các yêu cầu này chủ yếu là luật hướng dẫn LLM. Code hiện tại có kiểm tra danh sách tag hợp lệ và bằng chứng Regex, nhưng chưa có bước kiểm tra độc lập, tổng quát để đảm bảo mọi cặp tag mâu thuẫn đều bị loại.

---

## 3. Đây là luật cứng hay luật mềm?

* **Luật mềm (LLM):** LLM hiểu ngữ nghĩa của mô tả và đề xuất nhãn dựa trên prompt. LLM có thể suy luận trực tiếp từ bằng chứng, nhưng cũng có thể phân loại sai.
* **Luật cứng (Python):** `clean_tags()`, `apply_aliases()` và `validate_tags()` giới hạn tập nhãn, chuẩn hóa alias và loại các tag không có mẫu bằng chứng phù hợp.

Đây là một cách thiết kế khá thực tế: dùng LLM để xử lý sự đa dạng ngôn ngữ, sau đó dùng code để tăng tính nhất quán và giảm hallucination.

---

## 4. Những điểm nên cải thiện

Với bộ dữ liệu phòng trà Hà Nội, ba cải tiến ưu tiên gồm:

1. **Phân biệt “không có bằng chứng” với “có bằng chứng phủ định”:** Ví dụ “quán không tổ chức nhạc sống” vẫn chứa từ khóa “nhạc sống”; Regex hiện tại có thể giữ nhầm `co_live_music`.
2. **Bổ sung luật phủ định và ngữ cảnh:** Ví dụ “không có chỗ để xe” không được gán `parking`; “không hút thuốc” lại là bằng chứng hợp lệ cho `khong_thuoc_la`.
3. **Ghi lại bằng chứng cho từng tag:** Hiện đã có `--show-dropped` và `--explain` để kiểm tra tag bị loại hoặc lý do giữ. Có thể mở rộng để lưu cả câu trích dẫn làm bằng chứng, giúp kiểm toán và cải thiện taxonomy dễ hơn.

---

> **Kết luận:** `llm_tagger.py` sử dụng chiến lược LLM gán nhãn theo taxonomy cố định + Regex xác minh bằng chứng + alias normalization. Điểm mạnh là kiểm soát được tập nhãn và hạn chế tag vô căn cứ; điểm yếu lớn nhất là Regex chưa hiểu ngữ cảnh phủ định, hàm ý và các cách diễn đạt chưa có trong luật.

Luật 5: Hạn chế tag quá mức
Prompt yêu cầu LLM:
- Không đoán khi không có căn cứ.
- Không gán các tag mâu thuẫn, ví dụ sang_trong và binh_dan.
- Thường chỉ chọn 1–4 tag nổi bật cho mỗi quán.
- Trả về JSON đúng cấu trúc.
Lưu ý rằng các yêu cầu này chủ yếu là luật hướng dẫn LLM. Code hiện tại có kiểm tra danh sách tag hợp lệ và bằng chứng Regex, nhưng chưa có bước kiểm tra độc lập, tổng quát để đảm bảo mọi cặp tag mâu thuẫn đều bị loại.

## 8. Dựng văn bản embedding

Trong `src/embeddings/embedder.py`, hàm `build_text()` ghép:

1. Tên quán.
2. Khu vực/quận.
3. Tag đã chuyển từ mã chuẩn hóa sang nhãn dễ đọc.
4. Nội dung mô tả của từng nguồn.

Ví dụ tag `nhac_trinh` được chuyển thành “nhạc Trịnh”. Cách viết này giúp văn bản có ngữ cảnh thay vì chỉ nhúng một đoạn mô tả rời rạc. Hàm `normalize_text()` chuẩn hóa Unicode NFC, bỏ khoảng trắng đầu/cuối và chuyển chữ thường.

In [ ]:
TAG_LABELS = {
    "music_genres": {
        "nhac_trinh": "nhạc Trịnh", "acoustic": "acoustic", "jazz": "jazz",
        "bolero": "bolero", "lofi_chill": "lofi chill", "indie": "indie",
        "traditional": "nhạc truyền thống", "piano": "piano", "pop_ballad": "pop ballad",
    },
    "atmosphere_tags": {
        "yen_tinh": "yên tĩnh", "lang_man": "lãng mạn", "co_kinh": "cổ kính",
        "view_dep": "view đẹp", "san_vuon": "sân vườn", "rooftop": "rooftop",
        "sang_trong": "sang trọng", "binh_dan": "bình dân", "tre_trung": "trẻ trung",
        "co_dien": "cổ điển",
    },
    "features": {
        "co_live_music": "có nhạc sống", "cho_dat_cho": "có thể đặt chỗ",
        "do_an_nhe": "có đồ ăn nhẹ", "parking": "có chỗ đỗ xe",
        "khong_thuoc_la": "không hút thuốc", "wifi_manh": "wifi mạnh",
    },
}
TAG_PREFIX = {
    "music_genres": "Thể loại nhạc",
    "atmosphere_tags": "Không gian",
    "features": "Tiện ích",
}

def normalize_text(text: str) -> str:
    return unicodedata.normalize("NFC", text).strip().lower()

def build_text(name, district, tags, description, include_tags=True):
    parts = [f"{name}."]
    if district:
        parts.append(f"Khu vực: {district}.")
    if include_tags:
        for tag_type, prefix in TAG_PREFIX.items():
            values = tags.get(tag_type) or []
            if values:
                labels = ", ".join(TAG_LABELS[tag_type].get(v, v) for v in values)
                parts.append(f"{prefix}: {labels}.")
    parts.append(description.strip())
    return " ".join(parts)

for venue in venues[:2]:
    print(build_text(venue["name"], venue["district"], venue, venue["descriptions"][0]["text"]))
    print()

## 9. Một point cho mỗi mô tả, không phải mỗi quán

Trong `src/embeddings/build_index.py`, mỗi dòng `tea_room_descriptions` trở thành một point riêng. Các mô tả của cùng quán dùng chung `tea_room_id`, nhưng có `description_id`, `source_name`, `source_url` và `text` riêng.

Lợi ích: truy vấn có thể khớp với cách diễn đạt cụ thể từ một nguồn. Sau đó `src/search/searcher.py` dùng `group_by="tea_room_id"` và `group_size=1` để mỗi quán chỉ xuất hiện một lần trong danh sách kết quả.

Với demo, mỗi mô tả là một point. Vì vậy tổng số point thường lớn hơn tổng số quán.

In [ ]:
documents = []
point_id = 1

for venue in venues:
    for description in venue["descriptions"]:
        text_for_embedding = build_text(
            name=venue["name"],
            district=venue["district"],
            tags=venue,
            description=description["text"],
            include_tags=True,
        )
        documents.append({
            "point_id": point_id,
            "venue_id": venue["id"],
            "name": venue["name"],
            "district": venue["district"],
            "address": venue["address"],
            "price_min": venue["price_min"],
            "price_max": venue["price_max"],
            "hours": venue["hours"],
            "music_genres": venue["music_genres"],
            "atmosphere_tags": venue["atmosphere_tags"],
            "features": venue["features"],
            "source_name": description["source"],
            "text": description["text"],
            "embedding_text": text_for_embedding,
        })
        point_id += 1

print("Số point:", len(documents))
print("Ví dụ văn bản embedding:")
print(documents[0]["embedding_text"])

## 10. Sinh vector embedding

`SentenceTransformer.encode()` chuyển văn bản thành vector số. Trong code dự án, vector được chuẩn hóa (`normalize_embeddings=True`) và collection dùng khoảng cách cosine. Lab cũng chuẩn hóa vector để tương thích với cách làm đó.

Vector là biểu diễn số của nội dung; nó không phải bản dịch của câu hỏi và cũng không tự bảo đảm kết quả đúng. Chất lượng phụ thuộc model, dữ liệu, cách tạo text và cách đánh giá.

In [ ]:
embedding_texts = [d["embedding_text"] for d in documents]
t0 = time.time()
vectors = embedder.encode(
    [normalize_text(t) for t in embedding_texts],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)
print("Shape:", vectors.shape)
print("Thời gian:", round(time.time() - t0, 2), "giây")
print("Norm vector đầu tiên:", round(float(np.linalg.norm(vectors[0])), 4))

## 11. Tạo collection Qdrant in-memory

Qdrant hỗ trợ chạy local trong bộ nhớ thông qua `QdrantClient(":memory:")`. Đây là lựa chọn phù hợp cho lab: không cần Docker hoặc server ngoài, dữ liệu mất khi kernel dừng và không ảnh hưởng collection production.

Collection dùng cosine distance. Payload giữ metadata phục vụ hiển thị và lọc; vector giữ biểu diễn embedding.

In [ ]:
COLLECTION = "phong_tra_colab_demo"
qdrant = QdrantClient(":memory:")

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=int(vectors.shape[1]),
        distance=models.Distance.COSINE,
    ),
)

points = []
for doc, vector in zip(documents, vectors):
    payload = {k: doc[k] for k in [
        "venue_id", "name", "district", "address", "price_min", "price_max",
        "hours", "music_genres", "atmosphere_tags", "features", "source_name", "text"
    ]}
    points.append(models.PointStruct(
        id=doc["point_id"],
        vector=vector.tolist(),
        payload=payload,
    ))

qdrant.upsert(collection_name=COLLECTION, points=points, wait=True)
print("Collection:", qdrant.get_collection(COLLECTION))

## 12. Tìm kiếm dense vector

Quy trình truy vấn:

1. Nhận câu hỏi tiếng Việt.
2. Chuẩn hóa và embedding câu hỏi bằng cùng model.
3. Tìm các vector gần nhất trong Qdrant.
4. Áp dụng bộ lọc metadata nếu có.
5. Gộp kết quả theo `venue_id` để mỗi quán chỉ xuất hiện một lần.

Trong hệ thống thực tế, `src/search/searcher.py` dùng `query_points_groups` để group theo `tea_room_id`; lab thực hiện group sau khi truy vấn để dễ quan sát kết quả.

## 12.1. Hybrid Search: Dense + Sparse + RRF

Trong project, Hybrid Search không chỉ là dense search kèm bộ lọc metadata. Hệ thống lấy kết quả từ **hai nhánh truy hồi độc lập**:

1. **Dense retrieval**: mã hóa truy vấn bằng cùng mô hình SentenceTransformer với dữ liệu, sau đó tìm vector gần nhất theo cosine.
2. **Sparse retrieval**: token hóa truy vấn và tìm các từ/cụm từ trùng khớp. Trong code project, `src/embeddings/sparse.py` tự tạo sparse vector; Qdrant áp dụng IDF qua `Modifier.IDF`.
3. **RRF fusion**: hợp nhất thứ hạng của hai nhánh bằng Reciprocal Rank Fusion, không cộng trực tiếp cosine score với sparse score vì hai loại điểm không cùng thang đo.
4. **Gộp theo quán**: nhiều description của cùng một quán không nên chiếm nhiều vị trí trong danh sách cuối.

Mã nguồn liên quan:
- `src/embeddings/sparse.py`: bỏ dấu, token hóa âm tiết đơn và cặp âm tiết liền kề, tạo token ID ổn định bằng CRC32 và trọng số TF.
- `src/database/indexer.py`: collection dùng vector tên `dense` và sparse vector tên `bm25`, có cấu hình IDF.
- `src/search/searcher.py`: chạy hai nhánh và gọi `rrf_fuse()` với `RRF_K = 60`.

**Lưu ý:** cell thực hành phía dưới dùng dữ liệu giả lập và Qdrant in-memory. Nó minh họa cùng nguyên lý, không phải benchmark trên dữ liệu phòng trà thật.

### 12.2. Công thức RRF và cách đọc điểm

Với một quán \(d\), điểm RRF được tính bằng tổng đóng góp từ thứ hạng ở mỗi nhánh:

\[
\operatorname{RRF}(d)=\sum_{r \in R}\frac{1}{k+\operatorname{rank}_r(d)}
\]

Trong đó `rank` bắt đầu từ 1, còn `k=60` trong code project. Nếu quán đứng hạng cao ở cả dense và sparse, điểm RRF sẽ được cộng đóng góp từ cả hai nhánh.

Trong `src/search/searcher.py`, điểm RRF được chia cho điểm cao nhất có thể để đưa về khoảng dễ đọc hơn. **Điểm này không phải xác suất đúng và không thể so sánh trực tiếp với cosine score của dense-only search.**

Code project còn có một cổng chất lượng: nếu `min_score` được đặt (mặc định 0.45), kết quả hybrid phải thuộc tập quán đạt ngưỡng cosine ở nhánh dense. Điều này hạn chế kết quả chỉ lọt vào vì trùng vài từ khóa nhưng ngữ nghĩa không phù hợp.

### 12.3. Phạm vi demo và kiểm thử chữ hoa/thường

Các cell trước mục này đã xây dựng collection **dense-only**. Để không phá collection đó, phần lab Hybrid Search sẽ tạo một collection riêng tên `phong_tra_colab_hybrid_demo`, có vector `dense` và sparse vector `bm25`.

Dữ liệu trong notebook vẫn là dữ liệu giả lập. Nếu thay bằng dữ liệu thật, cần bảo đảm mỗi document có ID quán ổn định, mô tả, metadata và tag đã chuẩn hóa.

Kiểm thử riêng:
- `JAZZ`, `Jazz`, `jazz`: kiểm tra chữ hoa/thường.
- `Trinh`, `Trịnh`: kiểm tra bỏ dấu ở nhánh sparse; nhánh dense vẫn dùng embedding văn bản đã chuẩn hóa NFC và lowercase, không mặc định rằng có dấu/không dấu tạo ra cùng vector.
- “nhạc Trịnh yên tĩnh” và “đêm nhạc Trịnh Công Sơn trong không gian tĩnh lặng”: kiểm tra lợi ích bổ sung của dense semantic retrieval.

Điều kiện quận, giá và tag là **metadata filters**, không phải một nhánh retrieval thứ ba trong RRF. Các filter được áp dụng cho cả nhánh dense lẫn sparse trước khi hợp nhất.

In [ ]:
def search_demo(query, limit=5, district=None, max_price=None, tags=None, min_score=None):
    query = query.strip()
    if not query:
        raise ValueError("Câu truy vấn không được để trống")

    must = []
    if district:
        must.append(models.FieldCondition(
            key="district", match=models.MatchValue(value=district)
        ))
    if max_price is not None:
        must.append(models.FieldCondition(
            key="price_min", range=models.Range(lte=max_price)
        ))
    for tag in (tags or []):
        if tag in TAG_LABELS["music_genres"]:
            field = "music_genres"
        elif tag in TAG_LABELS["atmosphere_tags"]:
            field = "atmosphere_tags"
        elif tag in TAG_LABELS["features"]:
            field = "features"
        else:
            raise ValueError(f"Tag không hợp lệ: {tag}")
        must.append(models.FieldCondition(
            key=field, match=models.MatchValue(value=tag)
        ))

    query_vector = embedder.encode(
        [normalize_text(query)], normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False
    )[0].tolist()

    query_filter = models.Filter(must=must) if must else None
    hits = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        query_filter=query_filter,
        limit=max(limit * 4, limit),
        with_payload=True,
    ).points

    # Giữ mô tả có điểm cao nhất của mỗi quán.
    best_by_venue = {}
    for hit in hits:
        if min_score is not None and hit.score < min_score:
            continue
        vid = hit.payload["venue_id"]
        if vid not in best_by_venue or hit.score > best_by_venue[vid]["score"]:
            best_by_venue[vid] = {
                "score": float(hit.score),
                **hit.payload,
            }

    return sorted(best_by_venue.values(), key=lambda x: x["score"], reverse=True)[:limit]

def show_results(results):
    if not results:
        print("Không có kết quả phù hợp.")
        return
    for rank, item in enumerate(results, start=1):
        print(f"{rank}. score={item['score']:.3f} | {item['name']} | {item['district']}")
        print(f"   tags: {item['music_genres']} / {item['atmosphere_tags']}")
        print(f"   giá minh họa: {item['price_min']:,}–{item['price_max']:,} VNĐ")
        print(f"   mô tả khớp: {item['text']}")
        print()

## 13. Dựng sparse vector cho demo

Sparse vector trong project được tạo bằng code riêng, không cần thư viện BM25 bên ngoài. Cách làm:
- Chuẩn hóa Unicode, chuyển chữ thường và bỏ dấu tiếng Việt.
- Tách âm tiết đơn, đồng thời tạo cặp âm tiết liên tiếp như `nhac_trinh` hoặc `san_vuon`.
- Băm token thành ID ổn định bằng CRC32 (không dùng `hash()` mặc định vì ID có thể thay đổi giữa các tiến trình).
- Với văn bản, dùng trọng số TF bão hòa; với truy vấn, mỗi token duy nhất có trọng số 1.
- Qdrant tính IDF khi collection được cấu hình với `Modifier.IDF`.

Cell dưới đây là bản rút gọn tương thích với logic trong `src/embeddings/sparse.py`.

In [ ]:
import re
import zlib
import unicodedata
from collections import Counter

SPARSE_VECTOR_NAME = "bm25"
DENSE_VECTOR_NAME = "dense"
RRF_K = 60
K1 = 1.2
WORD_RE = re.compile(r"\w+", re.UNICODE)

def strip_vietnamese_accents(text: str) -> str:
    text = unicodedata.normalize("NFD", text.lower().replace("đ", "d"))
    return "".join(ch for ch in text if unicodedata.category(ch) != "Mn")

def sparse_tokenize(text: str) -> list[str]:
    normalized = unicodedata.normalize("NFC", text)
    words = WORD_RE.findall(strip_vietnamese_accents(normalized))
    # Âm tiết đơn + bigram âm tiết liền kề
    return words + [f"{a}_{b}" for a, b in zip(words, words[1:])]

def token_id(token: str) -> int:
    # CRC32 tạo ID ổn định giữa các lần chạy
    return zlib.crc32(token.encode("utf-8"))

def sparse_parts(text: str, is_query: bool = False):
    counts = Counter(token_id(t) for t in sparse_tokenize(text))
    indices = sorted(counts)
    if is_query:
        values = [1.0] * len(indices)
    else:
        values = [
            counts[idx] * (K1 + 1) / (counts[idx] + K1)
            for idx in indices
        ]
    return indices, values

def make_sparse_vector(text: str, is_query: bool = False):
    indices, values = sparse_parts(text, is_query=is_query)
    return models.SparseVector(indices=indices, values=values)

print("Ví dụ token:", sparse_tokenize("Phòng trà nhạc Trịnh yên tĩnh"))
print("Số token sparse:", len(sparse_parts("Phòng trà nhạc Trịnh yên tĩnh")[0]))

## 14. Tạo collection Hybrid riêng

Qdrant collection của phần dense trước đó chỉ khai báo một vector không tên. Hybrid cần collection khác có:
- `dense`: vector embedding ngữ nghĩa, cùng chiều với model đã chọn.
- `bm25`: sparse vector từ khóa; `Modifier.IDF` để Qdrant tính trọng số IDF.
- Payload: metadata của document để lọc và hiển thị.

Không thể chỉ thêm sparse vector vào collection dense-only đã tạo trước đó mà không thay đổi schema. Vì vậy ta tạo collection mới và nạp lại các documents vào collection này.

In [ ]:
HYBRID_COLLECTION = "phong_tra_colab_hybrid_demo"

# Xóa collection demo cũ nếu chạy lại cell, không ảnh hưởng collection dense-only.
existing = {c.name for c in qdrant.get_collections().collections}
if HYBRID_COLLECTION in existing:
    qdrant.delete_collection(HYBRID_COLLECTION)

qdrant.create_collection(
    collection_name=HYBRID_COLLECTION,
    vectors_config={
        DENSE_VECTOR_NAME: models.VectorParams(
            size=int(vectors.shape[1]),
            distance=models.Distance.COSINE,
        )
    },
    sparse_vectors_config={
        SPARSE_VECTOR_NAME: models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    },
)

hybrid_points = []
for doc, dense_vector in zip(documents, vectors):
    payload = {k: doc[k] for k in [
        "venue_id", "name", "district", "address", "price_min", "price_max",
        "hours", "music_genres", "atmosphere_tags", "features", "source_name", "text"
    ]}
    hybrid_points.append(models.PointStruct(
        id=doc["point_id"],
        vector={
            DENSE_VECTOR_NAME: dense_vector.tolist(),
            SPARSE_VECTOR_NAME: make_sparse_vector(doc["embedding_text"], is_query=False),
        },
        payload=payload,
    ))

qdrant.upsert(
    collection_name=HYBRID_COLLECTION,
    points=hybrid_points,
    wait=True,
)
print("Hybrid collection:", HYBRID_COLLECTION)
print("Số point:", qdrant.count(collection_name=HYBRID_COLLECTION, exact=True).count)

## 15. Hàm Hybrid Search: Dense + Sparse + RRF

Thứ tự thực hiện:
1. Chuẩn hóa truy vấn và tạo dense query vector.
2. Truy vấn dense và sparse độc lập, cùng áp dụng metadata filters.
3. Mỗi nhánh được gộp sơ bộ theo `venue_id`, giữ một mô tả đại diện cho mỗi quán.
4. Dùng RRF hợp nhất thứ hạng hai danh sách.
5. Nếu truyền `dense_min_score`, chỉ giữ quán xuất hiện trong tập dense đạt ngưỡng. Mặc định demo để `None`, vì ngưỡng 0.45 của project cần được kiểm định theo model và dữ liệu thực tế.

Trong project thật, `src/search/searcher.py` dùng pool rộng (`max(limit * 4, 30)`) và `RRF_K=60`. Demo cũng dùng pool rộng để giảm việc một quán có nhiều mô tả chiếm hết top-k.

In [ ]:
def build_demo_filter(district=None, max_price=None, tags=None):
    must = []
    if district:
        must.append(models.FieldCondition(
            key="district", match=models.MatchValue(value=district)
        ))
    if max_price is not None:
        must.append(models.FieldCondition(
            key="price_min", range=models.Range(lte=max_price)
        ))
    for tag in tags or []:
        if tag in TAG_LABELS["music_genres"]:
            field = "music_genres"
        elif tag in TAG_LABELS["atmosphere_tags"]:
            field = "atmosphere_tags"
        elif tag in TAG_LABELS["features"]:
            field = "features"
        else:
            raise ValueError(f"Tag không hợp lệ: {tag}")
        must.append(models.FieldCondition(
            key=field, match=models.MatchValue(value=tag)
        ))
    return models.Filter(must=must) if must else None

def rrf_fuse(rankings, k=RRF_K):
    scores = {}
    for ranking in rankings:
        for rank, venue_id in enumerate(ranking, start=1):
            scores[venue_id] = scores.get(venue_id, 0.0) + 1.0 / (k + rank)
    return scores

def best_per_venue(points, score_key="score"):
    best = {}
    for hit in points:
        payload = hit.payload
        venue_id = payload["venue_id"]
        score = float(hit.score)
        if venue_id not in best or score > best[venue_id][score_key]:
            best[venue_id] = {**payload, score_key: score}
    return best

def search_hybrid(
    query,
    limit=5,
    district=None,
    max_price=None,
    tags=None,
    dense_min_score=None,
    pool=None,
):
    query = query.strip()
    if not query:
        raise ValueError("Câu truy vấn không được để trống")

    query_filter = build_demo_filter(district, max_price, tags)
    pool = pool or max(limit * 4, 30)

    dense_query = embedder.encode(
        [normalize_text(query)],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )[0].tolist()

    dense_hits = qdrant.query_points(
        collection_name=HYBRID_COLLECTION,
        query=dense_query,
        using=DENSE_VECTOR_NAME,
        query_filter=query_filter,
        limit=pool * 3,
        with_payload=True,
    ).points

    sparse_query = make_sparse_vector(query, is_query=True)
    sparse_hits = []
    if sparse_query.indices:
        sparse_hits = qdrant.query_points(
            collection_name=HYBRID_COLLECTION,
            query=sparse_query,
            using=SPARSE_VECTOR_NAME,
            query_filter=query_filter,
            limit=pool * 3,
            with_payload=True,
        ).points

    dense_by_venue = best_per_venue(dense_hits, "dense_score")
    sparse_by_venue = best_per_venue(sparse_hits, "sparse_score")

    dense_rank = sorted(
        dense_by_venue,
        key=lambda vid: dense_by_venue[vid]["dense_score"],
        reverse=True,
    )[:pool]
    sparse_rank = sorted(
        sparse_by_venue,
        key=lambda vid: sparse_by_venue[vid]["sparse_score"],
        reverse=True,
    )[:pool]

    rankings = [r for r in (dense_rank, sparse_rank) if r]
    fused_scores = rrf_fuse(rankings)
    payload_by_venue = {**sparse_by_venue, **dense_by_venue}
    # dense_by_venue được ưu tiên để lấy đoạn mô tả đại diện có điểm dense cao nhất
    payload_by_venue.update(dense_by_venue)

    ordered_ids = sorted(fused_scores, key=fused_scores.get, reverse=True)
    if dense_min_score is not None:
        dense_allowed = {
            vid for vid, item in dense_by_venue.items()
            if item["dense_score"] >= dense_min_score
        }
        ordered_ids = [vid for vid in ordered_ids if vid in dense_allowed]

    best_possible = len(rankings) / (RRF_K + 1) if rankings else 1.0
    results = []
    for venue_id in ordered_ids[:limit]:
        item = payload_by_venue[venue_id].copy()
        item["score"] = fused_scores[venue_id] / best_possible
        item["rrf_score_raw"] = fused_scores[venue_id]
        results.append(item)
    return results

def show_hybrid_results(results):
    if not results:
        print("Không có kết quả phù hợp.")
        return
    for rank, item in enumerate(results, start=1):
        print(
            f"{rank}. RRF={item['score']:.3f} | "
            f"Dense={item.get('dense_score', float('nan')):.3f} | "
            f"{item['name']} | {item['district']}"
        )
        print(f"   Tag: {item['music_genres']} / {item['atmosphere_tags']}")
        print(f"   Nguồn demo: {item['source_name']}")
        print(f"   Mô tả: {item['text']}")
        print()

print("Đã định nghĩa search_hybrid()")

## 16. Chạy thử Hybrid Search

So sánh cùng một truy vấn giữa dense-only và hybrid. Điểm RRF không cùng thang đo với cosine, nên hãy so sánh thứ hạng và độ liên quan của kết quả, không so sánh trực tiếp hai cột điểm.

In [ ]:
test_query = "Tìm phòng trà nhạc Trịnh trong không gian yên tĩnh"

print("=== Dense-only ===")
show_results(search_demo(test_query, limit=5, min_score=None))

print("\\n=== Hybrid: Dense + Sparse + RRF ===")
show_hybrid_results(search_hybrid(test_query, limit=5))

### 16.1. Thử nghiệm từ khóa và chuẩn hóa chữ hoa/thường

Sparse branch bỏ dấu và chuyển chữ thường; dense branch chuẩn hóa Unicode NFC và chữ thường trước khi embed. Bởi vậy, kiểm thử `JAZZ`/`Jazz`/`jazz` và `Trinh`/`Trịnh` riêng biệt sẽ giúp quan sát khác biệt giữa hai nhánh.

Kết quả có thể giống nhau trên tập demo nhỏ; điều đó không chứng minh mọi truy vấn thực tế đều cho cùng thứ hạng.

In [ ]:
case_queries = ["JAZZ", "Jazz", "jazz", "Trinh", "Trịnh"]

for q in case_queries:
    print("=" * 80)
    print("QUERY:", q)
    show_hybrid_results(search_hybrid(q, limit=3))

### 16.2. Hybrid Search kết hợp metadata filters

Bộ lọc quận, giá và tag được áp dụng cho cả dense retrieval lẫn sparse retrieval trước khi fusion. Các điều kiện trong ví dụ là AND: quán phải thỏa tất cả điều kiện được truyền vào. Trong demo, `price_min` thiếu hoặc không thỏa điều kiện sẽ không xuất hiện khi đặt `max_price`.

In [ ]:
filtered_query = "nhạc acoustic thư giãn"
filtered_results = search_hybrid(
    filtered_query,
    limit=5,
    district="Cầu Giấy",
    max_price=100000,
    tags=["acoustic"],
)
print("Query:", filtered_query)
print("Filter: district=Cầu Giấy AND price_min<=100000 AND tag=acoustic")
show_hybrid_results(filtered_results)

## 17. Thử một số câu truy vấn Dense

Điểm cosine chỉ dùng để xếp hạng tương đối trong một lần tìm kiếm/model cụ thể; không nên diễn giải trực tiếp thành xác suất câu trả lời đúng. Ngưỡng `min_score` cần được chọn bằng dữ liệu đánh giá, không nên lấy mặc định của demo làm chuẩn production.

In [ ]:
queries = [
    "Tôi muốn nghe nhạc Trịnh trong không gian yên tĩnh",
    "Có nơi nào chơi jazz và piano vào buổi tối không?",
    "Quán có sân vườn và nhạc acoustic cho nhóm bạn",
]

for query in queries:
    print("=" * 90)
    print("QUERY:", query)
    show_results(search_demo(query, limit=3))

## 18. Lọc theo metadata

Tìm kiếm ngữ nghĩa trả lời câu hỏi “nội dung nào gần ý nghĩa truy vấn?”. Metadata filtering trả lời điều kiện có cấu trúc như quận, giá tối đa hoặc tag cụ thể.

Trong code dự án, `build_filter()` áp dụng các điều kiện theo AND. Khi lọc giá, logic hiện tại là `price_min_vnd <= max_price`; quán thiếu giá sẽ không thỏa điều kiện này. Các tag cũng được nối bằng AND, không phải OR.

In [ ]:
query = "Tìm nơi nghe nhạc nhẹ nhàng, thư giãn"
results = search_demo(
    query,
    limit=5,
    district="Cầu Giấy",
    max_price=100000,
    tags=["acoustic"],
)
print("Query:", query)
print("Filter: district=Cầu Giấy AND price_min<=100000 AND tag=acoustic")
show_results(results)

## 19. Đánh giá đơn giản: Hit@K

Một demo tìm kiếm cần được đánh giá bằng truy vấn có nhãn liên quan. Bên dưới là một bộ đánh giá minh họa nhỏ, chỉ kiểm tra liệu một ID kỳ vọng có xuất hiện trong top-K hay không.

**Không coi đây là benchmark của hệ thống thực tế**: tập dữ liệu giả lập rất nhỏ, nhãn do người viết lab đặt và model có thể khác model production. Khi có dữ liệu thật, hãy xây dựng bộ truy vấn đại diện (tên quán, nhạc, không gian, ngân sách, quận, câu hỏi kết hợp) và gán nhãn thủ công.

In [ ]:
evaluation_cases = [
    {"query": "nhạc Trịnh và không gian yên tĩnh", "expected_id": "demo-trinh-ca"},
    {"query": "đêm nhạc jazz piano", "expected_id": "demo-jazz"},
    {"query": "sân vườn, acoustic, nhóm bạn trẻ", "expected_id": "demo-acoustic"},
    {"query": "bolero và tình khúc xưa", "expected_id": "demo-bolero"},
]

def evaluate_hit_at_k(cases, k=3):
    rows = []
    for case in cases:
        results = search_demo(case["query"], limit=k, min_score=None)
        returned_ids = [r["venue_id"] for r in results]
        rows.append({
            "query": case["query"],
            "expected_id": case["expected_id"],
            "top_k_ids": returned_ids,
            f"hit@{k}": int(case["expected_id"] in returned_ids),
        })
    df = pd.DataFrame(rows)
    return df, float(df[f"hit@{k}"].mean()) if len(df) else 0.0

eval_df, hit_rate = evaluate_hit_at_k(evaluation_cases, k=3)
display(eval_df)
print(f"Hit@3 trên bộ demo: {hit_rate:.1%}")

## 20. Đọc kết quả và các giới hạn hiện tại

### Những gì lab đã thực hiện

- Sinh embedding từ văn bản có tên, quận, tag và mô tả.
- Lưu vector cùng payload vào Qdrant in-memory.
- Truy vấn dense vector, lọc metadata và gộp mô tả theo quán.
- Chạy một phép đánh giá Hit@K minh họa.

### Những gì lab chưa thực hiện

- Không crawl web trong notebook.
- Không kết nối SQL Server đang chạy trên máy cá nhân.
- Không gọi Gemini để làm sạch hoặc gán tag.
- Không sử dụng collection Qdrant production.
- Có triển khai sparse retrieval và RRF trong phần lab minh họa mới, nhưng dữ liệu vẫn là giả lập và collection in-memory.
- Chưa phải đánh giá chất lượng trên dữ liệu phòng trà thật.

### Đối chiếu với code dự án

Trong `project_review.zip`, `src/search/searcher.py` đã có hai nhánh dense/sparse và hàm `rrf_fuse()`. Các cell Hybrid Search bổ sung ở notebook là **bản demo độc lập**, mô phỏng logic cốt lõi trên dữ liệu giả lập; chúng không import trực tiếp code repository và không chứng minh kết quả production.

## 21. Cách chạy pipeline thực tế trên máy phát triển

Các lệnh sau được ghi trong `docs/COMMANDS.md`; chạy từ thư mục gốc repository và đảm bảo biến môi trường đã được cấu hình. Không chạy các lệnh có thể xóa dữ liệu nếu chưa sao lưu.

```bash
# Cài thư viện
uv sync

# Chạy unit tests
python -m pytest tests/unit -v

# Crawl theo config
python -m src.crawler.pipeline configs/crawler_config.yaml

# Hợp nhất dữ liệu
python -m src.processing.entity_resolver

# Làm sạch xác định
python -m src.processing.deterministic_cleaner

# Làm sạch bằng LLM (cần GEMINI_API_KEY)
python -m src.processing.llm_cleaner

# Gán tag (cần cấu hình provider/API nếu cache không đủ)
python -m src.processing.llm_tagger

# Nạp SQL Server (kiểm tra --help/schema trước khi dùng)
python -m src.database.load_sql --help

# Xem trước văn bản/token, không ghi Qdrant
python -m src.embeddings.build_index --dry-run

# Dựng lại collection và index theo code hiện tại
python -m src.embeddings.build_index

# Tìm kiếm
python -m src.search.searcher "quán nhạc Trịnh yên tĩnh"
```

**Cảnh báo:** `build_index.py` hiện tạo lại collection (`recreate=True`) sau khi embedding hoàn tất. Hãy hiểu rõ tác động trước khi chạy với dữ liệu đang phục vụ ứng dụng.

## 22. Kết nối dữ liệu thật vào lab

Nếu muốn thay dữ liệu giả lập bằng `tagged_entities.json`:

1. Tải file từ môi trường phát triển lên Colab (không upload `.env`).
2. Đọc JSON thành list các entity.
3. Chuyển mỗi entity và từng phần tử trong `descriptions` thành một document.
4. Giữ lại tag, quận, giá, ID quán và thông tin nguồn trong payload.
5. Chạy lại các cell embedding → Qdrant → search → evaluation.

Schema có thể khác giữa file JSON và SQL Server. Cần đối chiếu `src/database/sql_models.py` và `src/embeddings/build_index.py`; không nên giả định tên trường giống nhau hoàn toàn.

In [ ]:
# Mẫu đọc file JSON sau khi bạn upload vào Colab.
# Bỏ comment và sửa đường dẫn khi có file thật.
#
# from google.colab import files
# uploaded = files.upload()  # chọn tagged_entities.json
# input_path = next(iter(uploaded))
# with open(input_path, "r", encoding="utf-8") as f:
#     real_entities = json.load(f)
# print("Loaded entities:", len(real_entities))
# print("First keys:", list(real_entities[0].keys()) if real_entities else [])

## 23. Kết luận

Dự án không chỉ là gọi một model embedding. Chất lượng đầu ra phụ thuộc vào cả chuỗi: crawl đúng nội dung → hợp nhất đúng thực thể → làm sạch và gán tag có bằng chứng → tạo text phù hợp → lập chỉ mục → truy vấn và lọc → đánh giá trên câu hỏi thực tế.

Notebook này cung cấp lab có thể chạy độc lập để minh họa dense semantic search. Để trở thành tài liệu nghiệm thu/portfolio cuối cùng, nên bổ sung ảnh chụp kết quả trên dữ liệu thật, thống kê số quán/mô tả sau từng bước, và bộ truy vấn đánh giá được gán nhãn thủ công.

## 24. Lưu ý về số lượng bản ghi, quán và vector

Không được coi các con số sau là tương đương:

- **Bản ghi nguồn:** số mục thu thập được từ các website trước khi gộp.
- **Quán/entity chuẩn:** số thực thể còn lại sau entity resolution.
- **Description/chunk:** số đơn vị nội dung được giữ để tìm kiếm.
- **Vector point:** số point được lập chỉ mục; nếu mỗi description là một point, số point có thể lớn hơn số quán.

Các số liệu thay đổi theo lần crawl, cấu hình parser, quy tắc merge và dữ liệu đầu vào. Một bản ghi cũ có nhắc 64 bản ghi nguồn, 24 quán và 64 vector là ảnh chụp của một trạng thái trước đó, không nên dùng làm số liệu hiện hành. Tra cứu thống kê từ output của pipeline ở commit mới nhất trước khi đưa vào báo cáo.

Với thông tin được trao đổi gần đây, dự án đã có khoảng 25 địa điểm sau khi hợp nhất từ 6 website; hãy xác minh lại bằng dữ liệu đầu ra hiện tại trước khi đóng băng số liệu trong báo cáo.